# Quito TAZ and O-D Matrix — from official data

Traffic Analysis Zones and an origin-destination matrix built from the city's own published
data: the **barrio boundaries** the municipality uses, the **census sectors** and their counted
population, and the **cadastre** of buildings.

The aim is zones and a matrix that are recognisably Quito rather than a grid laid over it.

> **Data lives in `data/gov_data/` and is not tracked by git.** It is official DMQ material, not
> something this notebook downloads.

## A0. Imports and paths

`BASE_DIR` resolves whether the kernel starts in the repo root or in `notebooks/`. Maps go to
their own folder, emptied on each run so it only ever holds the current set.

In [ ]:
import gc
import logging
import os
import re
import textwrap
import unicodedata

import folium
from scipy.spatial import cKDTree
import geopandas as gpd
import numpy as np
import pandas as pd

_cwd = os.getcwd()
BASE_DIR = _cwd if os.path.isdir(os.path.join(_cwd, "outputs")) else os.path.dirname(_cwd)
DATA_DIR = os.path.join(BASE_DIR, "data")
GOV_DIR = os.path.join(DATA_DIR, "gov_data")
MAPS_DIR = os.path.join(BASE_DIR, "outputs", "taz_od_maps")

os.makedirs(MAPS_DIR, exist_ok=True)
for _stale in os.listdir(MAPS_DIR):
    _path = os.path.join(MAPS_DIR, _stale)
    if os.path.isfile(_path):
        os.remove(_path)

BARRIOS_SHP = os.path.join(GOV_DIR, "BARRIO_REF", "BARRIO_REF.shp")
POPULATION_XLSX = os.path.join(GOV_DIR, "03_Resultados_proyeccion2023_2035.xlsx")
for _needed in (BARRIOS_SHP, POPULATION_XLSX):
    if not os.path.exists(_needed):
        raise FileNotFoundError(f"Missing input: {_needed}")



def preview(frame, title, rows=5, columns=None, width=26):
    """First rows of a table, so the data can be read rather than only summarised.

    Drops the geometry column, which prints as an unreadable coordinate dump.
    """
    view = frame.drop(columns="geometry", errors="ignore")
    if columns:
        view = view[columns]
    print(f"\n{title} - first {rows} of {len(frame):,} rows:")
    print(view.head(rows).to_string(index=False, max_colwidth=width))


print(f"Data:  {GOV_DIR}\nMaps:  {MAPS_DIR}")

## A1. The barrios

`BARRIO_REF` is the municipality's neighbourhood layer: one polygon per **barrio**, grouped into
**parroquias** (parishes). It covers the whole Distrito Metropolitano, which is far larger than
the urban core.

Four of its columns — `norte`, `sur`, `este`, `oeste` — are the legal boundary description in
prose, naming the features each side abuts. They are dropped here.

| column | what it holds |
|---|---|
| `id` | row number — **the only unique key** |
| `codigo` | the municipality's barrio code; **not unique**, two codes are each shared by two barrios |
| `parroquia` | the parish it belongs to, and the only link to the population data |
| `nombre` | barrio name; repeats across parroquias, 1,091 distinct for 1,303 barrios |
| `geometry` | the polygon |

So neither `codigo` nor `nombre` is safe as a key on its own.

In [ ]:
barrios = gpd.read_file(BARRIOS_SHP)
barrios = barrios.drop(columns=["norte", "sur", "este", "oeste"])

# Areas need a projected CRS; the file is in degrees.
METRE_CRS = barrios.estimate_utm_crs()
barrios["area_km2"] = barrios.to_crs(METRE_CRS).area / 1e6

print(f"{len(barrios):,} barrios in {barrios['parroquia'].nunique()} parroquias, "
      f"crs {barrios.crs.to_string()}")
print(f"Extent: lat {barrios.total_bounds[1]:.4f} to {barrios.total_bounds[3]:.4f}, "
      f"lon {barrios.total_bounds[0]:.4f} to {barrios.total_bounds[2]:.4f}")
print(f"Total area: {barrios['area_km2'].sum():,.0f} km2")
print(f"Geometry: {barrios.geom_type.value_counts().to_dict()}, "
      f"{(~barrios.to_crs(METRE_CRS).is_valid).sum()} invalid, {barrios.is_empty.sum()} empty")

print("\nBarrio area (km2):")
print(barrios["area_km2"].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).round(2).to_string())

_dupes = barrios[barrios.duplicated("codigo", keep=False)]
print(f"\n{len(_dupes)} rows share a `codigo` with another barrio: "
      f"{sorted(_dupes['codigo'].unique())}")

preview(barrios, "barrios")

## A2. Population

`03_Resultados_proyeccion2023_2035.xlsx` holds population **per parroquia**, in three sheets:

| Sheet | Contents |
|---|---|
| `poblacion_2022` | the census base: `pob_total_2022`, and `prop_hombre` / `prop_mujer`, the sex split as fractions |
| `modelo_exponencial` | 2023–2035 under one model: `pob_proy`, `hombre`, `mujer`, and `proy_quito`, the city total |
| `resultados_por_modelos` | the same years with one column per model, for comparison |

All three carry `area` (**Rural** or **Urbana**), `codigo_parroquia` (the national DPA code) and
`nombre_parroquia`. The shapefile has no code, so the join has to go through the name.

The unit matters: population is published for **65 parroquias**, while the geometry is 1,303
barrios. Getting from one to the other is the first modelling decision this notebook has to make.

The four models are **distributions under a controlled total**: they disagree about individual
parroquias by up to 84,000 people, and still sum to the same city figure every year. Choosing
between them changes where people are, not how many.

In [ ]:
population = pd.read_excel(POPULATION_XLSX, sheet_name="poblacion_2022")
projections = pd.read_excel(POPULATION_XLSX, sheet_name="resultados_por_modelos")

print(f"{len(population)} parroquias, "
      f"{population['pob_total_2022'].sum():,} people in 2022")
print(population.groupby("area")["pob_total_2022"].agg(["count", "sum"]).to_string())

print(f"\nProjection years: {projections['anio'].min()}-{projections['anio'].max()}, "
      f"models: {[c for c in projections.columns if c not in ('area', 'codigo_parroquia', 'nombre_parroquia', 'anio')]}")

_models = [c for c in projections.columns
           if c not in ("area", "codigo_parroquia", "nombre_parroquia", "anio")]
_totals = projections.groupby("anio")[_models].sum()
print("\nCity total under each model - identical, because they distribute a fixed total:")
print(_totals.loc[[2023, 2029, 2035]].map(lambda v: f"{v:,.0f}").to_string())
_spread = (projections[_models].max(axis=1) - projections[_models].min(axis=1)).max()
print(f"\nLargest disagreement between models, one parroquia-year: {_spread:,.0f} people")

preview(population, "poblacion_2022")
preview(projections, "resultados_por_modelos")

## A3. Joining them

The two sources name parroquias differently in three ways: accents (`PINTAG` / `PÍNTAG`), spacing
(`LLANO  CHICO` with two spaces) and the leading article (`CONCEPCION` / `LA CONCEPCION`).
Matching the raw strings pairs 58 of 65; normalising all three pairs all 65.

In [ ]:
def canonical(name):
    """A parroquia name reduced to a joinable key: uppercase, unaccented, single-spaced, no article."""
    text = unicodedata.normalize("NFKD", str(name)).upper()
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = re.sub(r"\s+", " ", text).strip()
    return re.sub(r"^(LA|EL|LAS|LOS) ", "", text)


barrios["par_key"] = barrios["parroquia"].map(canonical)
population["par_key"] = population["nombre_parroquia"].map(canonical)

_raw = len(set(barrios["parroquia"]) & set(population["nombre_parroquia"]))
_keyed = len(set(barrios["par_key"]) & set(population["par_key"]))
print(f"Parroquias matched on the raw name: {_raw} of {barrios['parroquia'].nunique()}")
print(f"Parroquias matched on the key:      {_keyed} of {barrios['par_key'].nunique()}")

barrios = barrios.merge(
    population[["par_key", "codigo_parroquia", "area", "pob_total_2022"]],
    on="par_key", how="left", validate="many_to_one")
if barrios["pob_total_2022"].isna().any():
    raise RuntimeError("Some barrios did not pick up a population figure.")

_per_parroquia = barrios.groupby("par_key").size()
print(f"\nEvery barrio carries its parroquia's population. Barrios per parroquia: "
      f"min {_per_parroquia.min()}, median {int(_per_parroquia.median())}, max {_per_parroquia.max()}")
print("It is still a parroquia total, not a barrio one - splitting it is the step after this.")

# `par_key` is the join key; the three columns after it came from the spreadsheet.
preview(barrios, "barrios, joined",
        columns=["id", "nombre", "parroquia", "par_key", "area_km2", "area", "pob_total_2022"])

## A4. Parroquias

There is no parroquia file — the layer is the barrios **dissolved** by their `parroquia`
field. That works because the barrios tile the district exactly: no overlaps, no gaps, and the
dissolve returns 65 single-part polygons with no holes.

Population is published at this level, so a parroquia carries a real figure rather than a
shared-out one. Density is the first hint of how uneven the split to barrios will have to be.

Two columns are derived here rather than read: `n_barrios`, the number of barrios dissolved
into each parroquia, and `density`, population over area.

In [ ]:
parroquias = barrios.dissolve(
    by="par_key",
    aggfunc={"parroquia": "first", "area": "first", "pob_total_2022": "first",
             "area_km2": "sum", "id": "count"},
).rename(columns={"id": "n_barrios"}).reset_index()
parroquias["density"] = parroquias["pob_total_2022"] / parroquias["area_km2"]

# The dissolve is only trustworthy if the barrios really do tile; check rather than assume.
_parts = [1 if x.geom_type == "Polygon" else len(x.geoms) for x in parroquias.geometry]
_barrio_area = barrios.to_crs(METRE_CRS).area.sum()
_dissolved_area = parroquias.to_crs(METRE_CRS).area.sum()
print(f"{len(parroquias)} parroquias, {sum(_parts)} polygon parts "
      f"({(np.array(_parts) > 1).sum()} multi-part)")
print(f"Area preserved by the dissolve: "
      f"{_barrio_area / 1e6:,.1f} km2 -> {_dissolved_area / 1e6:,.1f} km2")

print("\nDensity, people per km2:")
print(parroquias.groupby("area")["density"].describe()[["count", "min", "50%", "max"]]
      .round(0).to_string())

print("\nDensest and emptiest:")
_cols = ["parroquia", "n_barrios", "area_km2", "pob_total_2022", "density"]
print(parroquias.nlargest(5, "density")[_cols].round(1).to_string(index=False))
print(parroquias.nsmallest(3, "density")[_cols].round(1).to_string(index=False))

preview(parroquias.round(1), "parroquias")

## A5. What they look like

Coloured by **area**, because that is the property that decides whether they can serve as zones.
The distribution is very skewed — a median of half a square kilometre against a largest of 166 —
so the classes are quintiles rather than equal steps.

In [ ]:
AREA_RAMP = ("#cde2fb", "#9ec5f4", "#5598e7", "#2a78d6", "#104281")   # one hue, light to dark
SIMPLIFY_M = 25          # invisible at city zoom, and a twentieth of the file size


def map_barrios(gdf, parishes=None, filename="0_barrios.html"):
    """The barrio layer shaded by area, with the parroquia borders over it.

    The two layers answer different questions - how big is a zone, and which zones share a
    population figure - so they are separate and can be toggled.
    """
    drawn = gdf.copy()
    drawn["geometry"] = drawn.to_crs(METRE_CRS).simplify(SIMPLIFY_M).to_crs("EPSG:4326")

    cuts = list(np.quantile(drawn["area_km2"], [0.2, 0.4, 0.6, 0.8]))
    labels = ([f"under {cuts[0]:.2f} km²"]
              + [f"{lo:.2f} &ndash; {hi:.2f} km²" for lo, hi in zip(cuts, cuts[1:])]
              + [f"over {cuts[-1]:.2f} km²"])

    def shade(area):
        for limit, colour in zip(cuts, AREA_RAMP):
            if area <= limit:
                return colour
        return AREA_RAMP[-1]

    drawn["fill"] = [shade(a) for a in drawn["area_km2"]]
    drawn["tip"] = [
        f'<div style="font-family:sans-serif;font-size:12px"><b>{r.nombre}</b><br>'
        f'parroquia: {r.parroquia}<br>area: {r.area_km2:.2f} km²<br>'
        f'parroquia population 2022: {r.pob_total_2022:,}<br>id: {r.id}</div>'
        for r in drawn.itertuples()]

    whole = drawn.geometry.union_all().centroid
    m = folium.Map(location=[whole.y, whole.x], zoom_start=11, tiles=None)
    # A pale backdrop, kept out of the layer control: it is not something to switch off.
    folium.TileLayer("CartoDB positron", name="Clean map", control=False).add_to(m)

    barrio_layer = folium.FeatureGroup(name="Barrios, by area", show=True).add_to(m)
    folium.GeoJson(
        drawn[["tip", "fill", "geometry"]],
        style_function=lambda f: {"fillColor": f["properties"]["fill"], "color": "#ffffff",
                                  "weight": 0.4, "fillOpacity": 0.75},
        highlight_function=lambda f: {"weight": 2, "color": "#111111"},
        tooltip=folium.GeoJsonTooltip(fields=["tip"], labels=False, sticky=True),
    ).add_to(barrio_layer)

    if parishes is not None:
        edges = parishes.copy()
        edges["geometry"] = edges.to_crs(METRE_CRS).simplify(SIMPLIFY_M).to_crs("EPSG:4326")
        edges["tip"] = [
            f'<div style="font-family:sans-serif;font-size:12px"><b>{r.parroquia}</b><br>'
            f'{r.n_barrios} barrios &middot; {r.area}<br>area: {r.area_km2:.1f} km²<br>'
            f'population 2022: {r.pob_total_2022:,}<br>'
            f'density: {r.density:,.0f} per km²</div>'
            for r in edges.itertuples()]
        parish_layer = folium.FeatureGroup(name="Parroquia borders", show=True).add_to(m)
        folium.GeoJson(
            edges[["tip", "geometry"]],
            # No fill at rest: the borders are the point, and a fill would hide the barrios.
            style_function=lambda f: {"fillOpacity": 0, "color": "#1a1a19", "weight": 2},
            # Hovering *tints* the parroquia rather than restroking it. Neighbours share every
            # internal border and each draws its own copy, so a restroke lands under whichever
            # neighbour happens to be on top - the border would go red or half-red depending on
            # draw order. A fill belongs to one polygon only, so it cannot be overdrawn.
            highlight_function=lambda f: {"fillOpacity": 0.28, "fillColor": "#d03b3b",
                                          "color": "#1a1a19", "weight": 2},
            tooltip=folium.GeoJsonTooltip(fields=["tip"], labels=False, sticky=True),
        ).add_to(parish_layer)

    folium.LayerControl(collapsed=False).add_to(m)

    swatches = "".join(
        f'<div><span style="display:inline-block;width:14px;height:10px;background:{c};'
        f'margin:0 6px 1px 0;border:1px solid #fff;vertical-align:middle;"></span>{lab}</div>'
        for c, lab in zip(AREA_RAMP, labels))
    m.get_root().html.add_child(folium.Element(
        '<div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:white;'
        'padding:9px 12px;border:1px solid #999;border-radius:6px;font-family:sans-serif;'
        f'font-size:13px;line-height:1.45;"><b>Barrio area</b>{swatches}'
        '<div style="margin-top:5px;color:#555;font-size:11px;">quintiles &middot; darker = larger'
        '<br>black outlines = parroquias, the level population is published at'
        '</div></div>'))
    m.get_root().html.add_child(folium.Element(
        '<div style="position:fixed;top:20px;right:20px;z-index:9999;background:white;'
        'padding:9px 12px;border:1px solid #999;border-radius:6px;font-family:sans-serif;'
        f'font-size:13px;line-height:1.5;"><b>Barrios and parroquias</b>'
        f'<div>{len(drawn):,} barrios in {drawn["parroquia"].nunique()} parroquias</div>'
        f'<div>{drawn["area_km2"].sum():,.0f} km² total</div>'
        f'<div style="color:#555;">median {drawn["area_km2"].median():.2f} km², '
        f'largest {drawn["area_km2"].max():.0f} km²</div></div>'))

    path = os.path.join(MAPS_DIR, filename)
    m.save(path)
    print(f"Map saved: {filename}\n  location: {os.path.abspath(path)}")
    return m


map_barrios(barrios, parroquias)

## A6. Where the people are

The same district shaded by **population density** rather than size. It is drawn at parroquia
level because that is the level the population is published at — shading each barrio would
repeat its parroquia's figure and imply a precision the data does not have.

A different hue from the first map on purpose: two sequential scales showing different
quantities should not look alike.

The barrio outlines stay available underneath, so you can see how many zones each shade covers.

In [ ]:
DENSITY_RAMP = ("#f9c3a8", "#f2996f", "#eb6834", "#b74a1f", "#8a3714")   # one hue, light to dark


def map_density(parishes, barrio_outlines=None, filename="1_density.html"):
    """Parroquias shaded by people per km2, with the barrio mosaic available beneath.

    Density spans four orders of magnitude - 4 people per km2 in Lloa against 16,000 in Solanda -
    so the classes are quintiles. Equal steps would put 64 of the 65 parroquias in one band.
    """
    drawn = parishes.copy()
    drawn["geometry"] = drawn.to_crs(METRE_CRS).simplify(SIMPLIFY_M).to_crs("EPSG:4326")

    cuts = list(np.quantile(drawn["density"], [0.2, 0.4, 0.6, 0.8]))
    labels = ([f"under {cuts[0]:,.0f}"]
              + [f"{lo:,.0f} &ndash; {hi:,.0f}" for lo, hi in zip(cuts, cuts[1:])]
              + [f"over {cuts[-1]:,.0f}"])

    def shade(value):
        for limit, colour in zip(cuts, DENSITY_RAMP):
            if value <= limit:
                return colour
        return DENSITY_RAMP[-1]

    drawn["fill"] = [shade(d) for d in drawn["density"]]
    drawn["tip"] = [
        f'<div style="font-family:sans-serif;font-size:12px"><b>{r.parroquia}</b><br>'
        f'{r.area} &middot; {r.n_barrios} barrios<br>'
        f'population 2022: {r.pob_total_2022:,}<br>'
        f'area: {r.area_km2:.1f} km²<br>'
        f'<b>{r.density:,.0f} people per km²</b></div>'
        for r in drawn.itertuples()]

    whole = drawn.geometry.union_all().centroid
    m = folium.Map(location=[whole.y, whole.x], zoom_start=11, tiles=None)
    # A pale backdrop, kept out of the layer control: it is not something to switch off.
    folium.TileLayer("CartoDB positron", name="Clean map", control=False).add_to(m)

    if barrio_outlines is not None:
        mosaic = barrio_outlines.copy()
        mosaic["geometry"] = mosaic.to_crs(METRE_CRS).simplify(SIMPLIFY_M).to_crs("EPSG:4326")
        under = folium.FeatureGroup(name="Barrio outlines", show=False).add_to(m)
        folium.GeoJson(
            mosaic[["geometry"]],
            # No fill at all, not merely a transparent one: an invisible fill still
            # catches the mouse, and this layer sits over the data that carries the panel.
            style_function=lambda f: {"fill": False, "color": "#52514e", "weight": 0.4},
        ).add_to(under)

    shaded = folium.FeatureGroup(name="Parroquias, by density", show=True).add_to(m)
    folium.GeoJson(
        drawn[["tip", "fill", "geometry"]],
        style_function=lambda f: {"fillColor": f["properties"]["fill"], "color": "#ffffff",
                                  "weight": 0.8, "fillOpacity": 0.65},
        # Hover deepens the fill rather than restroking: neighbours share every internal border
        # and each draws its own copy, so a restroke would land under whichever is on top.
        highlight_function=lambda f: {"fillOpacity": 1.0},
        tooltip=folium.GeoJsonTooltip(fields=["tip"], labels=False, sticky=True),
    ).add_to(shaded)
    folium.LayerControl(collapsed=False).add_to(m)

    swatches = "".join(
        f'<div><span style="display:inline-block;width:14px;height:10px;background:{c};'
        f'margin:0 6px 1px 0;border:1px solid #fff;vertical-align:middle;"></span>{lab}</div>'
        for c, lab in zip(DENSITY_RAMP, labels))
    m.get_root().html.add_child(folium.Element(
        '<div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:white;'
        'padding:9px 12px;border:1px solid #999;border-radius:6px;font-family:sans-serif;'
        f'font-size:13px;line-height:1.45;"><b>People per km²</b>{swatches}'
        '<div style="margin-top:5px;color:#555;font-size:11px;">quintiles &middot; darker = denser'
        '<br>parroquia level, the level the population is published at</div></div>'))

    _urban = drawn[drawn["area"].str.upper().str.startswith("URB")]
    m.get_root().html.add_child(folium.Element(
        '<div style="position:fixed;top:20px;right:20px;z-index:9999;background:white;'
        'padding:9px 12px;border:1px solid #999;border-radius:6px;font-family:sans-serif;'
        f'font-size:13px;line-height:1.5;"><b>Population density by parroquia</b>'
        f'<div>{drawn["pob_total_2022"].sum():,} people in {len(drawn)} parroquias</div>'
        f'<div>{len(_urban)} urban parroquias hold '
        f'{100 * _urban["pob_total_2022"].sum() / drawn["pob_total_2022"].sum():.0f}% of them</div>'
        f'<div style="color:#555;">on {100 * _urban["area_km2"].sum() / drawn["area_km2"].sum():.0f}%'
        f' of the land</div></div>'))

    path = os.path.join(MAPS_DIR, filename)
    m.save(path)
    print(f"Map saved: {filename}\n  location: {os.path.abspath(path)}")
    return m


map_density(parroquias, barrios)

## B1. Census sectors

The parroquia figures used so far are the finest the spreadsheet goes. The census itself is
published on a much smaller unit: `sector_anonimizado_a.shp` cuts the district into **7,179
sectors**, each with its own count.

| column | what it holds |
|---|---|
| `sec_anm` | sector code, unique; the first six digits are the parroquia's DPA code |
| `pob_t` | population |
| `v_pres` | dwellings present |
| `p_hog` | people per household |
| `serv_b` / `pob_serv_b` | share of dwellings / people with basic services |
| `parroquia`, `nom_par`, `adm_zonal` | the parroquia and zonal administration it sits in |

This removes the problem the notebook was heading towards. Splitting a parroquia total across its
barrios would have been an estimate; these are counts.

In [ ]:
SECTORS_SHP = os.path.join(GOV_DIR, "sector_censal_2022_inec_q", "sector_anonimizado_a.shp")
if not os.path.exists(SECTORS_SHP):
    raise FileNotFoundError(f"Missing input: {SECTORS_SHP}")

sectors = gpd.read_file(SECTORS_SHP)
# Two sectors in Guamani are empty: no people, no dwellings, and so no household size.
sectors[["pob_t", "p_hog"]] = sectors[["pob_t", "p_hog"]].fillna(0)
sectors["area_km2"] = sectors.to_crs(METRE_CRS).area / 1e6
sectors["density"] = sectors["pob_t"] / sectors["area_km2"]

print(f"{len(sectors):,} sectors, CRS {sectors.crs.to_string()}, "
      f"{sectors['area_km2'].sum():,.0f} km2 "
      f"(the barrios covered {barrios['area_km2'].sum():,.0f})")
print(f"Population: {sectors['pob_t'].sum():,.0f}, against "
      f"{parroquias['pob_total_2022'].sum():,} in the spreadsheet")
print(f"Sectors with no census figure: {(sectors['pob_t'] == 0).sum()}")

Two things are worth checking before trusting them as zones: that the sectors carry similar
numbers of people, and that they agree with the parroquia figures already in the notebook.

In [ ]:
_q = sectors["pob_t"].quantile([0, 0.1, 0.5, 0.9, 1])
print("People per sector - they are drawn to hold similar numbers:")
print(f"  min {_q[0]:.0f} | 10th {_q[0.1]:.0f} | median {_q[0.5]:.0f} | "
      f"90th {_q[0.9]:.0f} | max {_q[1]:.0f}")
print(f"  area km2: median {sectors['area_km2'].median():.3f}, "
      f"largest {sectors['area_km2'].max():.0f}")

# The sector file and the spreadsheet share the DPA parroquia code, so no name cleaning is needed.
_by_par = sectors.groupby(sectors["parroquia"].astype(str))["pob_t"].sum()
_sheet = population.set_index(population["codigo_parroquia"].astype(str))["pob_total_2022"]
_gap = (_by_par - _sheet).dropna()
print(f"\nParroquias matched on the DPA code alone: {len(_gap)} of {len(_sheet)}")
print(f"  city totals differ by {_by_par.sum() - _sheet.sum():,.0f}, but a single parroquia by "
      f"up to {_gap.abs().max():,.0f} people ({100 * (_gap.abs() / _sheet).max():.1f}%)")
_names = population.set_index(population["codigo_parroquia"].astype(str))["nombre_parroquia"]
_worst = _gap.abs().nlargest(3)
print("  the three that disagree most, in people: "
      + ", ".join(f"{_names[c]} {v:,.0f}" for c, v in _worst.items()))

# Sectors and barrios come from different agencies, so they need not line up.
_centres = sectors[["id", "geometry"]].copy()
_centres["geometry"] = _centres.representative_point()
_inside = gpd.sjoin(_centres, barrios[["par_key", "geometry"]], how="left", predicate="within")
_agrees = (_inside["par_key"].values == sectors["nom_par"].map(canonical).values).mean()
print(f"\nSector centres landing in a barrio of the same parroquia: {100 * _agrees:.1f}%")

preview(sectors.round(2), "census sectors",
        columns=["sec_anm", "nom_par", "pob_t", "v_pres", "p_hog", "area_km2", "density"])

## B2. The sectors as zones

Shaded by **population density** — the same quantity as the parroquia map, so the two can be read
against each other. This one goes far higher: a dense sector is a few blocks, while a parroquia
averages its dense core with the hillside behind it.

Because every sector holds roughly the same number of people, the map is largely a picture of
sector *size*. Small polygons are dense, large ones are empty.

In [ ]:
import json                                                          # noqa: E402
import shapely                                                       # noqa: E402

# ------------------------------------------------------------------------------------------
# The hover panel, one design shared by the maps below. The styling lives in a single
# stylesheet so the per-feature HTML stays small - at 7,179 sectors that matters.
# ------------------------------------------------------------------------------------------
TIP_CSS = """
<style>
.leaflet-tooltip { font-family: sans-serif; font-size: 12px; color: #222; }
.leaflet-tooltip table { margin: 0 !important; border-collapse: collapse; }
.leaflet-tooltip td, .leaflet-tooltip th { padding: 0; border: none; font-size: 12px; }
/* labels: a <th> when folium builds the rows, the first <td> when tip_html does */
.leaflet-tooltip th,
.leaflet-tooltip td:first-child { color: #555; font-weight: normal; text-align: left;
                                  padding-right: 12px; }
.leaflet-tooltip td:last-child { text-align: right; white-space: nowrap; }
/* the parroquia leads folium's rows, so give it the weight a title would have */
.leaflet-tooltip tr:first-child th + td { font-weight: bold; }
.leaflet-tooltip b { display: block; margin-bottom: 4px; white-space: nowrap; }
.leaflet-tooltip b span { font-weight: normal; color: white; border-radius: 3px;
                          padding: 1px 5px; font-size: 10px; margin-left: 6px; }
</style>
"""


def takes_white_text(hex_color, minimum=3.0):
    """Whether white text on this colour clears a contrast ratio, so a tag can be filled with it.

    Worked out rather than listed, so changing a colour cannot leave an unreadable tag behind.
    """
    channels = [int(hex_color[i:i + 2], 16) / 255 for i in (1, 3, 5)]
    linear = [c / 12.92 if c <= 0.04045 else ((c + 0.055) / 1.055) ** 2.4 for c in channels]
    luminance = 0.2126 * linear[0] + 0.7152 * linear[1] + 0.0722 * linear[2]
    return 1.05 / (luminance + 0.05) >= minimum


def tip_html(title, rows, tag=None, tag_color="#52514e"):
    """The panel as an HTML string. `rows` is a list of (label, value) pairs, pre-formatted.

    Deliberately bare markup: this string is stored once per feature and its quotes are escaped
    again inside the GeoJSON, so anything beyond the content is paid for thousands of times.
    """
    if tag and not takes_white_text(tag_color):
        tag_color = "#52514e"
    chip = f'<span style=background:{tag_color}>{tag}</span>' if tag else ""
    body = "".join(f"<tr><td>{k}</td><td>{v}</td></tr>" for k, v in rows)
    return f"<b>{title}{chip}</b><table>{body}</table>"


def panel_tooltip():
    """The tooltip binding every map below uses; the panel itself is in the `tip` column."""
    return folium.GeoJsonTooltip(fields=["tip"], labels=False, sticky=True)


SECTOR_SIMPLIFY_M = 12       # sectors are a few hundred metres across, so a lighter hand than 25
COORD_GRID = 1e-5            # about 1 m: past what the map can show, and half the file size


def map_sectors(cells, parishes=None, barrio_outlines=None, filename="2_sectors.html"):
    """Census sectors shaded by people per km2, with the coarser layers available beneath.

    Quintiles again, and for the same reason as the parroquia map: the range spans several orders
    of magnitude, so equal steps would put almost everything in the bottom band.
    """
    drawn = cells.copy()
    drawn["geometry"] = drawn.to_crs(METRE_CRS).simplify(SECTOR_SIMPLIFY_M).to_crs("EPSG:4326")
    # Full float coordinates are two thirds of the page weight at this many polygons. Snapping
    # to a shared grid keeps neighbours sharing their borders exactly.
    drawn["geometry"] = shapely.set_precision(drawn.geometry.values, COORD_GRID)

    cuts = list(np.quantile(drawn["density"], [0.2, 0.4, 0.6, 0.8]))
    labels = ([f"under {cuts[0]:,.0f}"]
              + [f"{lo:,.0f} &ndash; {hi:,.0f}" for lo, hi in zip(cuts, cuts[1:])]
              + [f"over {cuts[-1]:,.0f}"])

    def shade(value):
        for limit, colour in zip(cuts, DENSITY_RAMP):
            if value <= limit:
                return colour
        return DENSITY_RAMP[-1]

    drawn["fill"] = [shade(d) for d in drawn["density"]]
    # Every "<" folium writes into the GeoJSON becomes six characters, so at this many features
    # the panel is built from columns and the labels are held once by the tooltip, not per row.
    drawn["f_par"] = drawn["nom_par"]
    drawn["f_sec"] = drawn["sec_anm"].astype(str)
    drawn["f_pop"] = drawn["pob_t"].map("{:,.0f}".format)
    drawn["f_dwe"] = drawn["v_pres"].map("{:,.0f}".format)
    drawn["f_hog"] = drawn["p_hog"].map("{:.1f}".format)
    drawn["f_ha"] = (drawn["area_km2"] * 100).map("{:,.1f} ha".format)
    drawn["f_den"] = drawn["density"].map("{:,.0f}".format)
    PANEL = ["f_par", "f_sec", "f_pop", "f_dwe", "f_hog", "f_ha", "f_den"]
    PANEL_LABELS = ["parroquia", "sector", "people", "dwellings", "per household",
                    "area", "people per km²"]

    whole = drawn.geometry.union_all().centroid
    m = folium.Map(location=[whole.y, whole.x], zoom_start=11, tiles=None)
    # A pale backdrop, kept out of the layer control: it is not something to switch off.
    folium.TileLayer("CartoDB positron", name="Clean map", control=False).add_to(m)
    m.get_root().header.add_child(folium.Element(TIP_CSS))

    if barrio_outlines is not None:
        mosaic = barrio_outlines.copy()
        mosaic["geometry"] = mosaic.to_crs(METRE_CRS).simplify(SIMPLIFY_M).to_crs("EPSG:4326")
        folium.GeoJson(
            mosaic[["geometry"]],
            style_function=lambda f: {"fill": False, "color": "#52514e", "weight": 0.4},
        ).add_to(folium.FeatureGroup(name="Barrio outlines", show=False).add_to(m))

    shaded = folium.FeatureGroup(name="Census sectors, by density", show=True).add_to(m)
    folium.GeoJson(
        drawn[PANEL + ["fill", "geometry"]],
        style_function=lambda f: {"fillColor": f["properties"]["fill"], "color": "#ffffff",
                                  "weight": 0.25, "fillOpacity": 0.8},
        # Same reason as the parroquia map: hover tints the fill instead of restroking, because
        # neighbours share every border and a restroke lands under whichever is drawn on top.
        highlight_function=lambda f: {"fillOpacity": 1.0, "weight": 1.5, "color": "#1a1a19"},
        tooltip=folium.GeoJsonTooltip(fields=PANEL, aliases=PANEL_LABELS, sticky=True),
    ).add_to(shaded)

    if parishes is not None:
        borders = parishes.copy()
        borders["geometry"] = borders.to_crs(METRE_CRS).simplify(SIMPLIFY_M).to_crs("EPSG:4326")
        folium.GeoJson(
            borders[["geometry"]],
            style_function=lambda f: {"fill": False, "color": "#1a1a19", "weight": 1.1},
        ).add_to(folium.FeatureGroup(name="Parroquia borders", show=True).add_to(m))

    folium.LayerControl(collapsed=False).add_to(m)

    swatches = "".join(
        f'<div><span style="display:inline-block;width:14px;height:10px;background:{c};'
        f'margin:0 6px 1px 0;border:1px solid #fff;vertical-align:middle;"></span>{lab}</div>'
        for c, lab in zip(DENSITY_RAMP, labels))
    m.get_root().html.add_child(folium.Element(
        '<div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:white;'
        'padding:9px 12px;border:1px solid #999;border-radius:6px;font-family:sans-serif;'
        f'font-size:13px;line-height:1.45;"><b>People per km²</b>{swatches}'
        '<div style="margin-top:5px;color:#555;font-size:11px;">quintiles &middot; darker = denser'
        '<br>census sector level, about 350 people each</div></div>'))

    m.get_root().html.add_child(folium.Element(
        '<div style="position:fixed;top:20px;right:20px;z-index:9999;background:white;'
        'padding:9px 12px;border:1px solid #999;border-radius:6px;font-family:sans-serif;'
        f'font-size:13px;line-height:1.5;"><b>Population density by census sector</b>'
        f'<div>{drawn["pob_t"].sum():,.0f} people in {len(drawn):,} sectors</div>'
        f'<div>median sector: {drawn["area_km2"].median() * 100:.1f} ha, '
        f'{drawn["pob_t"].median():.0f} people</div>'
        f'<div style="color:#555;">the parroquia map shows the same figure in '
        f'{0 if parishes is None else len(parishes)} shapes</div></div>'))

    path = os.path.join(MAPS_DIR, filename)
    m.save(path)
    print(f"Map saved: {filename}  ({os.path.getsize(path) / 1e6:.1f} MB)"
          f"\n  location: {os.path.abspath(path)}")
    return m


map_sectors(sectors, parroquias)

## B3. What this means for the zone system

The sectors are the right *kind* of unit — real counts, small enough that a zone has one walk
distance to a stop rather than several. There are simply too many of them: an O-D matrix grows
with the square of the zone count, and 7,179 zones is 51 million pairs.

So the next step is aggregation, not disaggregation — grouping sectors into a few hundred zones
that respect barrio and parroquia boundaries. The building and land-use data comes in there too,
to say what each zone attracts rather than only how many people it holds.

In [ ]:
print("Zone pairs at each level, across the district:")
for _label, _n in [("census sectors", len(sectors)), ("barrios", len(barrios)),
                   ("parroquias", parroquias["parroquia"].nunique())]:
    print(f"  {_label:<30} {_n:>6,} zones -> {_n ** 2:>12,} pairs")

## C1. From sectors to zones

7,179 sectors is finer than a transport model can use. Spread a plausible peak hour over that
many zones and the average cell of the O-D matrix holds **0.017 trips** — the matrix is not
detailed, it is empty. Two further limits bite before size does: a zone should not be smaller
than the network can distinguish (many sectors share the same access stop), and it should not be
finer than the trip rates that will be applied to it.

Aggregating is safe in a way that splitting was not. Summing sectors into zones is arithmetic —
the population stays counted, just reported more coarsely.

The census code carries a ready-made grouping. `sec_anm` is hierarchical, and its first nine
digits are the INEC **census zona**: 749 of them, median 3,513 people and 0.43 km². Four rules
turn those into zones:

1. group sectors by census zona
2. split any zona that straddles a parroquia, so every zone sits in exactly one
3. split any zona that is not a single connected piece
4. merge zones under a minimum population, split zones over a maximum

### The choices that are arbitrary

Four of the numbers below are judgement, not data. They are collected here so they can be moved.

| choice | set to | what moving it does |
|---|---|---|
| grouping base | census zona | barrios would give ~1,300 zones and ignore the census hierarchy |
| parroquia as a hard edge | on | off saves ~57 zones, but zones stop matching the published population |
| `MIN_ZONE_POP` | 500 | higher absorbs more small zones into their neighbours |
| `MAX_ZONE_POP` | 8,000 | lower splits more of the large rural zones |
| which neighbour a small zone joins | the one sharing the most sector borders | another rule picks different neighbours |
| centroid | population-weighted | a geometric centroid puts rural connectors on an empty hillside |

The first two shape the zone system. The last four only touch its tails: 68 zones fall under the
minimum and 5 over it, together 2.9% of the district's population.

In [ ]:
from scipy.sparse import coo_matrix                                     # noqa: E402
from scipy.sparse.csgraph import connected_components                   # noqa: E402
from sklearn.cluster import AgglomerativeClustering                     # noqa: E402

MIN_ZONE_POP = 500       # under this, a zone is merged into a neighbour
MAX_ZONE_POP = 8000      # over this, a zone is split into compact pieces

sectors = sectors.reset_index(drop=True)
sectors["row"] = np.arange(len(sectors))
sectors["zona"] = sectors["sec_anm"].astype(str).str[:9]
_sec_m = sectors.to_crs(METRE_CRS)

# Which sectors share a border. Everything downstream - connectedness, merging, splitting - runs
# on this one adjacency list rather than on geometry.
_pairs = gpd.sjoin(_sec_m[["row", "geometry"]], _sec_m[["row", "geometry"]],
                   predicate="touches", how="inner")
EDGES = _pairs[["row_left", "row_right"]].to_numpy()
EDGES = EDGES[EDGES[:, 0] != EDGES[:, 1]]
print(f"{len(sectors):,} sectors, {len(EDGES):,} adjacency pairs "
      f"({len(EDGES) / len(sectors):.1f} neighbours each)")

# Rules 1 and 2: the census zona, cut at parroquia borders.
_base, _ = pd.factorize(sectors["zona"] + "|" + sectors["parroquia"].astype(str))

# Rule 3: a zone has to be one piece, so split on the connected components of the adjacency graph.
_within = EDGES[_base[EDGES[:, 0]] == _base[EDGES[:, 1]]]
_graph = coo_matrix((np.ones(len(_within)), (_within[:, 0], _within[:, 1])),
                    shape=(len(sectors), len(sectors)))
_n_parts, _part = connected_components(_graph, directed=False)
sectors["zone"] = pd.Series(_base).astype(str) + "-" + pd.Series(_part).astype(str)


def zone_table(frame):
    """Zones summed up from their sectors. Population is added, never apportioned."""
    return frame.groupby("zone").agg(
        pop=("pob_t", "sum"), dwellings=("v_pres", "sum"), km2=("area_km2", "sum"),
        sectors=("row", "size"), parroquia=("parroquia", "first"), nom_par=("nom_par", "first"))


_t = zone_table(sectors)
print(f"After rules 1-3: {len(_t):,} zones "
      f"({(_t['pop'] < MIN_ZONE_POP).sum()} under {MIN_ZONE_POP:,}, "
      f"{(_t['pop'] > MAX_ZONE_POP).sum()} over {MAX_ZONE_POP:,})")

### Rule 4: the tails

Merging takes the smallest zone, absorbs it into the neighbour it shares the most sector borders
with, and repeats. A zone with no neighbour inside its own parroquia is left alone.

Splitting clusters an oversized zone's sectors into compact pieces. `AgglomerativeClustering`
with the adjacency matrix as its `connectivity` argument only ever merges neighbours, so every
piece comes out contiguous.

In [ ]:
_zone_of = sectors["zone"].to_numpy().copy()
_pop = sectors["pob_t"].to_numpy()
_par = sectors["parroquia"].astype(str).to_numpy()
_locked, _merged = set(), 0

while True:
    _sizes = pd.DataFrame({"zone": _zone_of, "pop": _pop, "par": _par}) \
               .groupby("zone").agg(pop=("pop", "sum"), par=("par", "first"))
    _small = _sizes[(_sizes["pop"] < MIN_ZONE_POP) & (~_sizes.index.isin(_locked))]
    if _small.empty:
        break
    _target = _small["pop"].idxmin()
    _left, _right = _zone_of[EDGES[:, 0]], _zone_of[EDGES[:, 1]]
    _cand = pd.Series(_right[(_left == _target) & (_right != _target)]).value_counts()
    _cand = _cand[[z for z in _cand.index if _sizes.loc[z, "par"] == _sizes.loc[_target, "par"]]]
    if _cand.empty:
        _locked.add(_target)              # an island in its own parroquia; nothing to join
        continue
    _zone_of[_zone_of == _target] = _cand.index[0]
    _merged += 1

sectors["zone"] = _zone_of
print(f"Merged {_merged} zones under {MIN_ZONE_POP:,} people"
      + (f"; {len(_locked)} had no neighbour in their parroquia and were kept" if _locked else ""))

_centres = np.column_stack([_sec_m.geometry.representative_point().x,
                            _sec_m.geometry.representative_point().y])
_big = zone_table(sectors).query("pop > @MAX_ZONE_POP")
_labels = sectors["zone"].to_numpy().copy()
for _zone, _row in _big.iterrows():
    _idx = np.where(sectors["zone"].to_numpy() == _zone)[0]
    _k = int(np.ceil(_row["pop"] / MAX_ZONE_POP))
    if len(_idx) < _k:
        continue
    _pos = {r: i for i, r in enumerate(_idx)}
    _sub = EDGES[np.isin(EDGES[:, 0], _idx) & np.isin(EDGES[:, 1], _idx)]
    _conn = coo_matrix((np.ones(len(_sub)),
                        ([_pos[a] for a in _sub[:, 0]], [_pos[b] for b in _sub[:, 1]])),
                       shape=(len(_idx), len(_idx)))
    _labels[_idx] = [f"{_zone}s{v}" for v in
                     AgglomerativeClustering(n_clusters=_k, connectivity=_conn,
                                             linkage="ward").fit_predict(_centres[_idx])]
sectors["zone"] = _labels
print(f"Split {len(_big)} zones over {MAX_ZONE_POP:,} people")

# The keys so far are build artefacts. Renumber them, by parroquia and then by sector code,
# so a zone id is stable between runs and readable in a tooltip.
_order = (sectors.groupby("zone")
          .agg(par=("parroquia", "first"), code=("sec_anm", "min"))
          .sort_values(["par", "code"]))
sectors["zone"] = sectors["zone"].map(
    {z: f"Z{i:04d}" for i, z in enumerate(_order.index, start=1)})
print(f"Zones numbered Z0001 to Z{len(_order):04d}")

## C2. The zones

The sectors are dissolved into zones, and the sector table keeps its `zone` label so anything
measured per sector later — floor area, land use — can be summed up without redoing this.

The centroid is **population-weighted**: the average sector centre weighted by the people in it.
In a rural zone the people sit in a village in one corner, and a geometric centroid would put the
connector on an empty hillside. Where the weighted point falls outside its own zone, which a
concave shape allows, it is moved to the nearest point inside.

In [ ]:
zones = sectors.dissolve(
    by="zone",
    aggfunc={"pob_t": "sum", "v_pres": "sum", "area_km2": "sum", "id": "count",
             "parroquia": "first", "nom_par": "first", "adm_zonal": "first"},
).rename(columns={"pob_t": "pop", "v_pres": "dwellings", "id": "n_sectors"}).reset_index()
zones["density"] = zones["pop"] / zones["area_km2"]

# Population-weighted centre, computed in metres and put back into lat/lon.
_pts = _sec_m.geometry.representative_point()
_w = sectors.assign(x=_pts.x, y=_pts.y, w=sectors["pob_t"].clip(lower=1e-9))
_wc = _w.groupby("zone").apply(
    lambda g: pd.Series({"cx": np.average(g["x"], weights=g["w"]),
                         "cy": np.average(g["y"], weights=g["w"])}),
    include_groups=False)
_centroids = gpd.GeoSeries(gpd.points_from_xy(_wc["cx"], _wc["cy"]), index=_wc.index,
                           crs=METRE_CRS)

_zones_m = zones.set_index("zone").to_crs(METRE_CRS)
_outside = ~_centroids.within(_zones_m.geometry)
if _outside.any():
    _centroids.loc[_outside] = _zones_m.geometry[_outside].representative_point()
zones["centroid"] = _centroids.to_crs("EPSG:4326").reindex(zones["zone"]).to_numpy()
print(f"Centroids moved inside their zone: {_outside.sum()} of {len(zones)}")

_parts = [1 if g.geom_type == "Polygon" else len(g.geoms) for g in zones.geometry]
print(f"\n{len(zones):,} zones, from {len(sectors):,} sectors")
print(f"  population preserved: {zones['pop'].sum():,.0f} of {sectors['pob_t'].sum():,.0f}")
print(f"  population  min {zones['pop'].min():,.0f} | 10th {zones['pop'].quantile(.1):,.0f} | "
      f"median {zones['pop'].median():,.0f} | 90th {zones['pop'].quantile(.9):,.0f} | "
      f"max {zones['pop'].max():,.0f}")
print(f"  area km2    median {zones['area_km2'].median():.2f} | "
      f"90th {zones['area_km2'].quantile(.9):.2f} | max {zones['area_km2'].max():.0f}")
print(f"  sectors per zone: median {zones['n_sectors'].median():.0f}, "
      f"max {zones['n_sectors'].max()}")
print(f"  zones spanning more than one parroquia: "
      f"{(sectors.groupby('zone')['parroquia'].nunique() > 1).sum()}")
print(f"  zones that are a single connected polygon: "
      f"{sum(1 for p in _parts if p == 1)} of {len(zones)}")

preview(zones.round(2), "zones",
        columns=["zone", "nom_par", "n_sectors", "pop", "dwellings", "area_km2", "density"])

## C3. What they look like

The same density scale as the two maps above, so the three can be read against each other: 65
parroquias, 7,179 sectors, and now the zones in between. The centroids are the points a transport
model would hang its connectors on.

In [ ]:
def map_zones(cells, parishes=None, sector_outlines=None, filename="3_zones.html"):
    """Zones shaded by people per km2, with their population-weighted centroids."""
    drawn = cells.copy()
    drawn["geometry"] = drawn.to_crs(METRE_CRS).simplify(SECTOR_SIMPLIFY_M).to_crs("EPSG:4326")
    drawn["geometry"] = shapely.set_precision(drawn.geometry.values, COORD_GRID)

    cuts = list(np.quantile(drawn["density"], [0.2, 0.4, 0.6, 0.8]))
    labels = ([f"under {cuts[0]:,.0f}"]
              + [f"{lo:,.0f} &ndash; {hi:,.0f}" for lo, hi in zip(cuts, cuts[1:])]
              + [f"over {cuts[-1]:,.0f}"])

    def shade(value):
        for limit, colour in zip(cuts, DENSITY_RAMP):
            if value <= limit:
                return colour
        return DENSITY_RAMP[-1]

    drawn["fill"] = [shade(d) for d in drawn["density"]]
    drawn["tip"] = [
        tip_html(r.nom_par, [("people", f"{r.pop:,.0f}"),
                             ("dwellings", f"{r.dwellings:,.0f}"),
                             ("census sectors", f"{r.n_sectors}"),
                             ("area", f"{r.area_km2:.2f} km²"),
                             ("people per km²", f"{r.density:,.0f}")],
                 tag=r.zone, tag_color="#2a78d6")
        for r in drawn.itertuples()]

    whole = drawn.geometry.union_all().centroid
    m = folium.Map(location=[whole.y, whole.x], zoom_start=11, tiles=None)
    # A pale backdrop, kept out of the layer control: it is not something to switch off.
    folium.TileLayer("CartoDB positron", name="Clean map", control=False).add_to(m)
    m.get_root().header.add_child(folium.Element(TIP_CSS))

    if sector_outlines is not None:
        mosaic = sector_outlines.copy()
        mosaic["geometry"] = (mosaic.to_crs(METRE_CRS).simplify(SECTOR_SIMPLIFY_M)
                              .to_crs("EPSG:4326"))
        mosaic["geometry"] = shapely.set_precision(mosaic.geometry.values, COORD_GRID)
        folium.GeoJson(
            mosaic[["geometry"]],
            style_function=lambda f: {"fill": False, "color": "#52514e", "weight": 0.25},
        ).add_to(folium.FeatureGroup(name="Census sectors", show=False).add_to(m))

    shaded = folium.FeatureGroup(name="Zones, by density", show=True).add_to(m)
    folium.GeoJson(
        drawn[["tip", "fill", "geometry"]],
        style_function=lambda f: {"fillColor": f["properties"]["fill"], "color": "#ffffff",
                                  "weight": 0.5, "fillOpacity": 0.8},
        highlight_function=lambda f: {"fillOpacity": 1.0, "weight": 1.8, "color": "#1a1a19"},
        tooltip=panel_tooltip(),
    ).add_to(shaded)

    if parishes is not None:
        borders = parishes.copy()
        borders["geometry"] = borders.to_crs(METRE_CRS).simplify(SIMPLIFY_M).to_crs("EPSG:4326")
        folium.GeoJson(
            borders[["geometry"]],
            style_function=lambda f: {"fill": False, "color": "#1a1a19", "weight": 1.1},
        ).add_to(folium.FeatureGroup(name="Parroquia borders", show=True).add_to(m))

    dots = folium.FeatureGroup(name="Zone centroids", show=True).add_to(m)
    for r in cells.itertuples():
        folium.CircleMarker([r.centroid.y, r.centroid.x], radius=1.6, color="#1a1a19",
                            weight=0, fill=True, fill_opacity=0.85).add_to(dots)
    folium.LayerControl(collapsed=False).add_to(m)

    swatches = "".join(
        f'<div><span style="display:inline-block;width:14px;height:10px;background:{c};'
        f'margin:0 6px 1px 0;border:1px solid #fff;vertical-align:middle;"></span>{lab}</div>'
        for c, lab in zip(DENSITY_RAMP, labels))
    m.get_root().html.add_child(folium.Element(
        '<div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:white;'
        'padding:9px 12px;border:1px solid #999;border-radius:6px;font-family:sans-serif;'
        f'font-size:13px;line-height:1.45;"><b>People per km²</b>{swatches}'
        '<div style="margin-top:5px;color:#555;font-size:11px;">quintiles &middot; darker = denser'
        '<br>black dots are the population-weighted centroids</div></div>'))

    m.get_root().html.add_child(folium.Element(
        '<div style="position:fixed;top:20px;right:20px;z-index:9999;background:white;'
        'padding:9px 12px;border:1px solid #999;border-radius:6px;font-family:sans-serif;'
        f'font-size:13px;line-height:1.5;"><b>Traffic analysis zones</b>'
        f'<div>{len(drawn):,} zones from {drawn["n_sectors"].sum():,} census sectors</div>'
        f'<div>{drawn["pop"].sum():,.0f} people, median '
        f'{drawn["pop"].median():,.0f} per zone</div>'
        f'<div style="color:#555;">every zone lies inside one parroquia</div></div>'))

    path = os.path.join(MAPS_DIR, filename)
    m.save(path)
    print(f"Map saved: {filename}  ({os.path.getsize(path) / 1e6:.1f} MB)"
          f"\n  location: {os.path.abspath(path)}")
    return m


map_zones(zones, parroquias)

# D. What is built on the zones

The zones so far carry people. An O-D matrix also needs the other end of the trip — where the
jobs, schools and shops are. Two more datasets cover that.

**`ba003_uso_suelo_edificabilidad_a`** is the municipal land-use plan: 8,207 polygons that tile
the district exactly, each with a use and the floors it may build. **`catastro.gdb`** is the
cadastre, which records buildings in two layers: `UNIDAD_CONSTRUCTIVA`, 870,070 units with a
median footprint of 43 m², and `BLOQUE_CONSTRUCTIVO`, 77,043 blocks with a median of 133 m².

Both layers carry the same handful of columns:

| column | what it holds |
|---|---|
| `CODIGO` | the building's code within its lot; not unique across the district |
| `PISOS` | number of floors |
| `Shape_Length`, `Shape_Area` | **in square degrees**, so they are recomputed in metres here |

There is no use or ownership attribute on either — that is why the class has to come from the
land-use polygon a building stands in.

**The two are disjoint, and together they are the city's buildings** — confirmed by opening
both layers in QGIS. Only 2% of block area is shared with a unit, and that little is edge
effects. They are interleaved rather than in separate districts: 41% of blocks touch a unit,
median gap 3.2 m. So the cadastre records some structures one way and some the other, and
**both layers are added in full**.

Units alone reach 95% of census sectors, blocks alone 65%, together 99%. The 248 sectors only
the blocks reach hold **89,051 people** who would otherwise live in zones with no recorded
buildings at all.

The two do different jobs. The cadastre says **what is built**, the land-use plan says **what it
may be used for**. Floor area comes from the cadastre; the class label comes from the zoning
polygon the building stands in. Zoning is never used as a quantity — `pisos_ba` is what a plot is
allowed, not what is on it.

`uso_prin` is the field that classifies cleanly; `uso_gral` is inconsistent in this file, with
2,362 Equipamiento polygons carrying `uso_gral = R`. Its 18 codes collapse to five classes:

| class | codes | what it is |
|---|---|---|
| residential | RUM, RUB, RUA, RR, RRR | urban and rural housing, all densities |
| mixed/commercial | M, CSE | the mixed corridors and specialised commerce |
| facilities | E | *equipamiento*: schools, hospitals, markets, sports grounds, public offices |
| heritage core | PUP | the protected historic centre |
| industrial | IMI, IAI, IAR | industry, by impact class |

Everything else — ecological protection, natural resources, reserve land — carries almost no
floor area and is dropped.

In [ ]:
LAND_USE_SHP = os.path.join(GOV_DIR, "ba003_uso_suelo_edificabilidad_a",
                            "ba003_uso_suelo_edificabilidad_a.shp")
CADASTRE_GDB = os.path.join(GOV_DIR, "BLOQUE_CONSTRUCTIVO", "BLOQUE_CONSTRUCTIVO", "catastro.gdb")
for _needed in (LAND_USE_SHP, CADASTRE_GDB):
    if not os.path.exists(_needed):
        raise FileNotFoundError(f"Missing input: {_needed}")

USE_CLASS = {"RUM": "residential", "RUB": "residential", "RUA": "residential",
             "RR": "residential", "RRR": "residential",
             "M": "mixed/commercial", "CSE": "mixed/commercial", "M-IMI-IAI": "mixed/commercial",
             "E": "facilities", "PUP": "heritage core",
             "IMI": "industrial", "IAI": "industrial", "IAR": "industrial"}
CLASSES = ["residential", "mixed/commercial", "facilities", "heritage core", "industrial"]

land_use = gpd.read_file(LAND_USE_SHP).to_crs(METRE_CRS)


def read_buildings(layer):
    """One cadastre layer, with the floor area each building carries."""
    frame = gpd.read_file(CADASTRE_GDB, layer=layer).to_crs(METRE_CRS)
    frame["floors"] = frame["PISOS"].clip(lower=1)
    frame["ground_m2"] = frame.area                # the footprint, i.e. one floor of it
    frame["floor_m2"] = frame["ground_m2"] * frame["floors"]
    frame["source"] = layer
    # CODIGO and PISOS are kept as read, so the preview below shows the layer as it is
    # stored rather than only what is derived from it.
    return frame[["source", "CODIGO", "PISOS", "floors", "ground_m2", "floor_m2",
                  "geometry"]]


_units = read_buildings("UNIDAD_CONSTRUCTIVA")
_blocks = read_buildings("BLOQUE_CONSTRUCTIVO")
_sec_m = sectors.to_crs(METRE_CRS)
units = pd.concat([_units, _blocks], ignore_index=True)


def sectors_reached(buildings):
    """Which census sectors a building layer puts anything in."""
    pts = buildings[["geometry"]].copy()
    pts["geometry"] = pts.representative_point()
    hit = gpd.sjoin(pts, _sec_m[["row", "geometry"]], how="inner", predicate="within")
    return set(hit["row"].unique())

print(f"land use: {len(land_use):,} polygons covering {land_use.area.sum() / 1e6:,.0f} km2, "
      f"against {sectors['area_km2'].sum():,.0f} for the district")
_with_units = sectors_reached(_units)
_with_blocks = sectors_reached(_blocks)
_only_blocks = _with_blocks - _with_units
print(f"cadastre: {len(_units):,} construction units reaching {len(_with_units):,} sectors")
print(f"          {len(_blocks):,} blocks reaching {len(_with_blocks):,} sectors")
print(f"          together {len(_with_units | _with_blocks):,} of {len(sectors):,} sectors; "
      f"the {len(_only_blocks):,} only blocks reach hold "
      f"{sectors.loc[sectors['row'].isin(_only_blocks), 'pob_t'].sum():,.0f} people")
print(f"  {len(units):,} buildings, {units['ground_m2'].sum() / 1e6:,.1f} km2 of footprint "
      f"and {units['floor_m2'].sum() / 1e6:,.1f} km2 of floor area")
print(f"  floors per building: median {units['floors'].median():.0f}, "
      f"max {units['floors'].max():.0f}")

_no_floors = units["PISOS"].isna().sum()
if _no_floors:
    print(f"  {_no_floors} buildings have no floor count and carry no floor area")

preview(land_use[["uso_prin", "leyenda", "pisos_ba", "superf_ha"]], "land use")
_shown = ["CODIGO", "PISOS", "floors", "ground_m2", "floor_m2"]
preview(_units.round(1), "UNIDAD_CONSTRUCTIVA", columns=_shown)
preview(_blocks.round(1), "BLOQUE_CONSTRUCTIVO", columns=_shown)

## D1. The class is the plan's, not ours

Every building takes the `uso_prin` of the polygon it stands in, and nothing is adjusted after
that. No rule reassigns a building because it looks too big to be a house, and none splits a
mixed-use building between its floors.

That leaves visible artefacts, and they are left visible on purpose. Zoning is a **plan, not a
survey**: it records what land may be used for, so anything standing on residential-zoned land is
counted as housing whatever it actually is.

| where | what it looks like | what is really there |
|---|---|---|
| Iñaquito | 175 m² of housing per person | office towers on Múltiple land |
| Nayón, Cumbayá | one polygon of 26,476 m² zoned *Residencial urbano de baja densidad* | estates and plant nurseries |
| Mariscal Sucre | 209 m² per person | hotels, offices and bars above the shops |

Every one of those is a property of the source. Correcting them here would need a rule invented
rather than measured, and an invented rule is harder to argue with later than a visible artefact.
The fix is better data: the municipal cadastre records `destino económico` per *predio*, which
would replace this inference with observed use.

Each unit is reduced to a point inside itself, then labelled by the zoning polygon and the sector
it falls in. A point rather than the polygon keeps every unit counted exactly once, whatever it
straddles.

In [ ]:
_pts = units[["floor_m2", "ground_m2", "floors", "geometry"]].copy()
_pts["geometry"] = _pts.representative_point()

_tagged = gpd.sjoin(_pts, land_use[["uso_prin", "geometry"]], how="left", predicate="within")
_tagged = _tagged[~_tagged.index.duplicated(keep="first")].drop(columns="index_right")
_sec_m = sectors.to_crs(METRE_CRS)
_tagged = gpd.sjoin(_tagged, _sec_m[["row", "geometry"]], how="left", predicate="within")
_tagged = _tagged[~_tagged.index.duplicated(keep="first")]
print(f"units landing in a land-use polygon: {100 * _tagged['uso_prin'].notna().mean():.1f}%")
print(f"units landing in a census sector:    {100 * _tagged['row'].notna().mean():.1f}%")

_tagged = _tagged[_tagged["row"].notna()].copy()
_tagged["klass"] = _tagged["uso_prin"].map(USE_CLASS)
_tagged["zone"] = _tagged["row"].map(sectors.set_index("row")["zone"])

_dropped = _tagged["klass"].isna()
print(f"floor area on land with no built-use class: "
      f"{_tagged.loc[_dropped, 'floor_m2'].sum() / 1e6:.2f} km2 "
      f"({100 * _tagged.loc[_dropped, 'floor_m2'].sum() / _tagged['floor_m2'].sum():.1f}%), dropped")

floor_by_class = (_tagged[~_dropped]
                  .pivot_table(index="zone", columns="klass", values="floor_m2",
                               aggfunc="sum", fill_value=0.0)
                  .reindex(columns=CLASSES, fill_value=0.0))

# The three building frames hold 947,113 geometries between them and are finished with once
# the floor area is summed. Releasing them here keeps the rest of the notebook - which opens a
# road network of 116,697 links - inside a sensible amount of memory.
N_BUILDINGS = len(units)
del units, _units, _blocks, _pts, _tagged
gc.collect()
print(f"released the building geometries; {N_BUILDINGS:,} of them were read")
print("\nfloor area by class, whole district (km2):")
print((floor_by_class.sum() / 1e6).round(2).to_string())

## D2. Attaching it to the zones

Floor area by class, summed onto the zones. Both cadastre layers count in full — they record
different structures, not the same ones twice.

Three checks on the result:

- floor area per person
- floor area per dwelling
- residential floor area against population, **by parroquia**

The last is by parroquia because census sectors are drawn to hold equal population, so nothing
varies against them.

In [ ]:
zones = zones.drop(columns=[c for c in CLASSES + ["built_m2", "activity_m2", "dominant"]
                            if c in zones.columns])
zones = zones.merge(floor_by_class, left_on="zone", right_index=True, how="left")
zones[CLASSES] = zones[CLASSES].fillna(0.0)
zones["built_m2"] = zones[CLASSES].sum(axis=1)
zones["activity_m2"] = zones["built_m2"] - zones["residential"]
zones["dominant"] = zones[CLASSES].idxmax(axis=1).where(zones["built_m2"] > 0, "no buildings")
# Housing floor area over people. Around 30 m² is ordinary; a very low figure is a fault in
# the data rather than crowded housing, and the map below is drawn to expose it.
zones["m2_per_person"] = zones["residential"] / zones["pop"].replace(0, np.nan)

print(f"zones with built floor area: {(zones['built_m2'] > 0).sum():,} of {len(zones):,}")
print(f"floor area placed on zones: {zones['built_m2'].sum() / 1e6:,.1f} km2")

_pop, _dw = zones["pop"].sum(), zones["dwellings"].sum()
print(f"\nresidential floor area per person:   {zones['residential'].sum() / _pop:5.1f} m2")
print(f"residential floor area per dwelling: {zones['residential'].sum() / _dw:5.1f} m2")

_by_par = zones.groupby("nom_par")[["pop", "residential", "activity_m2"]].sum()
print(f"\ncorrelation of residential floor area with population, by parroquia: "
      f"{_by_par[['pop', 'residential']].corr().iloc[0, 1]:.3f}")

print("\nwhich class dominates, by zone count:")
print(zones["dominant"].value_counts().to_string())

print(f"\nzones at least 80% one class: "
      f"{(zones[CLASSES].max(axis=1) / zones['built_m2'].replace(0, np.nan) >= 0.8).sum():,}")
print(f"activity floor area is {100 * zones['activity_m2'].sum() / zones['built_m2'].sum():.0f}% "
      f"of all floor area, and sits in {(zones['activity_m2'] > 0).sum():,} zones")

_thin = zones[(zones["pop"] > 1000) & (zones["m2_per_person"] < 10)]
print(f"\nHousing floor area per person: median {zones['m2_per_person'].median():.1f} m²")
print(f"  but {len(_thin)} zones hold over 1,000 people on under 10 m² each - "
      f"{_thin['pop'].sum():,.0f} people, {100 * _thin['pop'].sum() / zones['pop'].sum():.0f}% "
      f"of the district")
print("  those are the zones whose floor area cannot be taken at face value:")
print("   " + _thin["dominant"].value_counts().to_string().replace("\n", "\n   "))

preview(zones.assign(**{c: zones[c].round(0) for c in CLASSES}), "zones with floor area",
        columns=["zone", "nom_par", "pop", "residential", "mixed/commercial", "facilities",
                 "industrial", "dominant"])

## D3. Where the activity is

Two views of the same zones, switched by the radio buttons.

- **Floor area that is not housing** — shops, offices, schools, workshops, industry, in m². A
  **total, not a share**. Drawn as circles rather than shading: the zones holding most of it
  cover very little land, so on a filled map they would be specks.
- **Housing per person** — a check, not an input. Productions come from the census.

Median housing per person is **38 m²**. **62 zones holding 193,370 people record under 10 m²**,
which is one of two faults:

- **nothing recorded** — El Quinche, Yaruquí, Amaguaña, Quitumbe. The cadastre is absent, so
  these zones produce trips and attract almost none.
- **recorded as something else** — Centro Histórico, Iñaquito. Homes above shops on Múltiple
  land count as commercial.

Read the red as *the classification is unreliable here*, not as how anyone lives.

In [ ]:
NOTHING_GREY = "#b8b8b3"          # no buildings, or nobody living there
# Housing is pale so the four activity classes carry the map, but not so pale that it merges
# with the grey used for "nothing here" - those two were 4.4 apart and are now 13.1.
CLASS_COLORS = {"residential": "#f0dcb0",
                "mixed/commercial": "#D55E00",
                "facilities": "#0072B2",
                "industrial": "#CC79A7",
                "heritage core": "#009E73",
                "no buildings": NOTHING_GREY}
PLAIN_FILL = "#eae7e0"            # the quiet backdrop the circles sit on
CIRCLE_FILL, CIRCLE_EDGE = "#6b3fa0", "#3d2159"
CIRCLE_MAX_PX = 17                # radius of the busiest zone; the rest scale by square root
# Blue against a red flag: the pair with the widest margin under colour-blind simulation of
# everything tried, at 17.4 where the floor is 6.
HOUSING_RAMP = ("#dbe7f6", "#adc8ea", "#7099d4", "#3f6bb0", "#1f3f78")
SUSPECT_RED = "#d03b3b"           # under 10 m² a person: a data fault, not a housing density
PLAUSIBLE_M2 = 10
ACT_LAYER = "Floor area that is not housing"
HPP_LAYER = "Housing per person"


def map_land_use(cells, parishes=None, filename="4_land_use.html"):
    """Zones by the floor area they hold that is not housing, and by housing per person."""
    drawn = cells.copy()
    drawn["geometry"] = drawn.to_crs(METRE_CRS).simplify(SECTOR_SIMPLIFY_M).to_crs("EPSG:4326")
    drawn["geometry"] = shapely.set_precision(drawn.geometry.values, COORD_GRID)

    # itertuples() rewrites any column name that is not a valid identifier, so rename first and
    # keep the mapping rather than guessing what it became.
    safe = {c: c.replace("/", "_").replace(" ", "_") for c in CLASSES}

    def rows_for(r):
        """Zone, people, then whatever classes it actually holds, then the non-housing total."""
        rows = [("zone", r.zone),
                ("people", f"{r.pop:,.0f}"),
                ("dwellings", f"{r.dwellings:,.0f}")]
        rows += [(c, f"{getattr(r, safe[c]) / 1000:,.1f}k m²")
                 for c in CLASSES if getattr(r, safe[c]) > 0]
        rows.append(("<b>not housing</b>",
                     f"<b>{r.activity_m2 / 1000:,.1f}k m²</b>"))
        if np.isfinite(r.m2_per_person):
            rows.append(("housing per person", f"{r.m2_per_person:,.0f} m²"))
        return rows

    drawn = drawn.rename(columns=safe)
    drawn["tip"] = [
        tip_html(r.nom_par, rows_for(r), tag=r.dominant,
                 tag_color=CLASS_COLORS[r.dominant])
        for r in drawn.itertuples()]

    # The second view says everything through the circles, so the zones behind them are a flat
    # backdrop - shading them by the same figure would state it twice.
    drawn["act_fill"] = PLAIN_FILL

    # Third view. The bands are quintiles of the zones that reach a believable figure, so the
    # ones that do not are a class of their own rather than the bottom of the scale.
    believable = drawn.loc[drawn["m2_per_person"] >= PLAUSIBLE_M2, "m2_per_person"]
    hpp_cuts = list(np.quantile(believable, [0.2, 0.4, 0.6, 0.8]))

    def housing_shade(value):
        if not np.isfinite(value):
            return NOTHING_GREY
        if value < PLAUSIBLE_M2:
            return SUSPECT_RED
        for limit, colour in zip(hpp_cuts, HOUSING_RAMP):
            if value <= limit:
                return colour
        return HOUSING_RAMP[-1]

    drawn["hpp_fill"] = [housing_shade(v) for v in drawn["m2_per_person"]]

    whole = drawn.geometry.union_all().centroid
    m = folium.Map(location=[whole.y, whole.x], zoom_start=11, tiles=None)
    # A pale backdrop, kept out of the layer control: it is not something to switch off.
    folium.TileLayer("CartoDB positron", name="Clean map", control=False).add_to(m)
    m.get_root().header.add_child(folium.Element(TIP_CSS))

    views = {}
    for name, column, first in [(ACT_LAYER, "act_fill", True),
                                (HPP_LAYER, "hpp_fill", False)]:
        # overlay=False makes these radio buttons: one view at a time, not stacked.
        group = folium.FeatureGroup(name=name, overlay=False, show=first).add_to(m)
        views[name] = group
        folium.GeoJson(
            drawn[["tip", column, "geometry"]],
            style_function=lambda f, c=column: {"fillColor": f["properties"][c],
                                                "color": "#ffffff", "weight": 0.5,
                                                "fillOpacity": 0.85 if c == "act_fill"
                                                else 0.55},
            highlight_function=lambda f: {"fillOpacity": 1.0, "weight": 1.8, "color": "#1a1a19"},
            tooltip=panel_tooltip(),
        ).add_to(group)

    # A circle whose area - not radius - is proportional to the floor area. The circles sit
    # over the zones, so they carry the same panel rather than swallowing the hover.
    biggest = drawn["activity_m2"].max()
    panels = dict(zip(drawn["zone"], drawn["tip"]))
    for r in cells.itertuples():
        if r.activity_m2 <= 0:
            continue
        folium.CircleMarker(
            [r.centroid.y, r.centroid.x],
            radius=CIRCLE_MAX_PX * (r.activity_m2 / biggest) ** 0.5,
            color=CIRCLE_EDGE, weight=0.6, fill=True, fill_color=CIRCLE_FILL,
            fill_opacity=0.75,
            tooltip=folium.Tooltip(panels[r.zone], sticky=True),
        ).add_to(views[ACT_LAYER])

    lift_borders = ""
    if parishes is not None:
        borders = parishes.copy()
        borders["geometry"] = borders.to_crs(METRE_CRS).simplify(SIMPLIFY_M).to_crs("EPSG:4326")
        # Straight onto the map rather than into a toggleable group: the borders orient the
        # reader and are wanted in both views, so there is nothing to switch off.
        border_layer = folium.GeoJson(
            borders[["geometry"]],
            style_function=lambda f: {"fill": False, "color": "#1a1a19", "weight": 1.1},
            control=False,
        ).add_to(m)
        # Changing view adds the incoming fills above them, so lift them back each time.
        lift_borders = f"{border_layer.get_name()}.bringToFront();"
    # Top left, under the zoom buttons: these layer names are longer than the info panel in
    # the top right, so the control would show from behind it.
    folium.LayerControl(collapsed=False, position="topleft").add_to(m)

    def swatch(colour, label):
        return (f'<div><span style="display:inline-block;width:14px;height:10px;'
                f'background:{colour};margin:0 6px 1px 0;border:1px solid #bbb;'
                f'vertical-align:middle;"></span>{label}</div>')

    BOX = ('position:fixed;bottom:24px;left:24px;z-index:9999;background:white;'
           'padding:9px 12px;border:1px solid #999;border-radius:6px;'
           'font-family:sans-serif;font-size:13px;line-height:1.45;')

    scale = "".join(
        f'<span style="display:inline-block;width:{2 * CIRCLE_MAX_PX * f:.0f}px;'
        f'height:{2 * CIRCLE_MAX_PX * f:.0f}px;border-radius:50%;background:{CIRCLE_FILL};'
        f'opacity:0.75;border:1px solid {CIRCLE_EDGE};vertical-align:middle;'
        f'margin-right:7px;"></span>'
        for f in (0.25, 0.55, 1.0))
    m.get_root().html.add_child(folium.Element(
        f'<div id="legend-act" style="{BOX}max-width:250px;">'
        f'<b>Floor area that is not housing</b>'
        f'<div style="margin:7px 0 2px;">{scale}</div></div>'))

    hpp_labels = ([f"{PLAUSIBLE_M2} &ndash; {hpp_cuts[0]:,.0f} m²"]
                  + [f"{lo:,.0f} &ndash; {hi:,.0f} m²" for lo, hi in zip(hpp_cuts, hpp_cuts[1:])]
                  + [f"over {hpp_cuts[-1]:,.0f} m²"])
    hpp = swatch(SUSPECT_RED, f"under {PLAUSIBLE_M2} m² &mdash; see below")
    hpp += "".join(swatch(c, lab) for c, lab in zip(HOUSING_RAMP, hpp_labels))
    m.get_root().html.add_child(folium.Element(
        f'<div id="legend-hpp" style="{BOX}display:none;max-width:265px;">'
        f'<b>Housing floor area per person</b>{hpp}'
        f'<div style="margin-top:5px;color:#555;font-size:11px;">About 30 m² is ordinary.'
        f' <b>Under 10 m² is not a real figure</b>: either the homes are classified as something'
        f' else, which happens in the centre and the mixed corridors, or the buildings are missing'
        f' from the cadastre, which happens in the newer periphery. Read the red as a fault in the'
        f' data, not as how anyone lives.</div></div>'))

    # Only one view is on screen at a time, so only one legend should be. folium appends the
    # map's own script while rendering, after this one, so wait for load before binding.
    legend_of = {ACT_LAYER: "legend-act", HPP_LAYER: "legend-hpp"}
    m.get_root().script.add_child(folium.Element(f"""
        window.addEventListener("load", function () {{
            var legends = {json.dumps(legend_of)};
            {m.get_name()}.on("baselayerchange", function (e) {{
                Object.keys(legends).forEach(function (name) {{
                    document.getElementById(legends[name]).style.display =
                        (name === e.name) ? "block" : "none";
                }});
                {lift_borders}
            }});
        }});
    """))

    m.get_root().html.add_child(folium.Element(
        '<div style="position:fixed;top:20px;right:20px;z-index:9999;background:white;'
        'padding:9px 12px;border:1px solid #999;border-radius:6px;font-family:sans-serif;'
        f'font-size:13px;line-height:1.5;"><b>Floor area by use</b>'
        f'<div>{drawn["built_m2"].sum() / 1e6:,.1f} km² over {len(drawn):,} zones</div>'
        f'<div>{100 * drawn["activity_m2"].sum() / drawn["built_m2"].sum():.0f}% of it is not '
        f'housing</div>'
        f'<div style="color:#555;">from {N_BUILDINGS:,} cadastral units</div></div>'))

    path = os.path.join(MAPS_DIR, filename)
    m.save(path)
    print(f"Map saved: {filename}  ({os.path.getsize(path) / 1e6:.1f} MB)"
          f"\n  location: {os.path.abspath(path)}")
    return m


map_land_use(zones, parroquias)

## D4. What is still missing

- **A rate.** Floor area is a stock, attractions are a flow. Trips per m² per hour, by class and
  purpose, is in no dataset here — so it will be an assumption.
- **Intensity of use.** A warehouse and an office of the same size attract very differently.

An establishment register or a household mobility survey would fix both. Neither is in
`data/gov_data`.

In [ ]:
print(f"{len(zones):,} zones, {zones['pop'].sum():,.0f} people, "
      f"{zones['built_m2'].sum() / 1e6:,.1f} km2 of floor area")
print(f"  activity floor area: {zones['activity_m2'].sum() / 1e6:,.1f} km2 "
      f"({100 * zones['activity_m2'].sum() / zones['built_m2'].sum():.0f}% of the total)")
print("\nThe ten zones that should attract the most trips:")
print(zones.nlargest(10, "activity_m2")[["zone", "nom_par", "pop", "activity_m2", "dominant"]]
      .assign(activity_m2=lambda d: (d["activity_m2"] / 1000).round(0))
      .rename(columns={"activity_m2": "activity_km2_x1000"}).to_string(index=False))

# E. The road network

Trips need a network twice over: a gravity model measures distance in travel time, and an
assignment needs somewhere to put the traffic.

The network is **downloaded once and kept** in `data/road_network/` — the district takes about
five and a half minutes to fetch and import. Delete that folder, or call
`build_road_network(rebuild=True)`, for a fresh copy.

> Two bugs in AequilibraE's OSM importer are worked around below rather than hidden. Both appear
> only at district scale.

In [ ]:
ASSUMED = {}


def assume(name, value, unit, why):
    """Record a number that is not in the data, and hand it back."""
    ASSUMED[name] = {"value": value, "unit": unit, "why": why}
    return value


def assumptions(width=88):
    """Every guess the model rests on, for reading and for replacing."""
    print(f"{len(ASSUMED)} values are assumed, not measured. Replace any of them and re-run.\n")
    for name, entry in ASSUMED.items():
        value = entry["value"]
        if isinstance(value, dict):
            head = ", ".join(f"{k}={v}" for k, v in list(value.items())[:3])
            shown = f"{head}, ... ({len(value)} classes)"
        else:
            shown = str(value)
        print(f"  {name}  =  {shown}")
        print(f"      {entry['unit']}")
        for line in textwrap.wrap(entry["why"], width - 6):
            print(f"      {line}")
        print()
    return pd.DataFrame(ASSUMED).T

In [ ]:
import shutil                                                          # noqa: E402

import shapely.wkb                                                    # noqa: E402
from shapely.geometry import LineString                                # noqa: E402

from aequilibrae import Parameters                                     # noqa: E402
from aequilibrae.project import Project                                # noqa: E402

ROAD_NETWORK_DIR = os.path.join(DATA_DIR, "road_network")


def _patch_osm_importer():
    """Make AequilibraE's OSM importer survive a district-sized download.

    Its importer casts node ids to int64 but leaves link ids alone, and across the district those
    arrive as tuples - 144,541 of them. That breaks the parquet cache it writes as a safety net,
    and then the SQLite insert. Both are the same fault at two call sites, and `osm_id` is not
    used for anything we model, so coercing it is safe.

    Returns the two originals so the patches can be lifted again afterwards.
    """
    real_parquet, real_records = pd.DataFrame.to_parquet, pd.DataFrame.to_records

    def safe_parquet(self, *args, **kwargs):
        frame = self
        if frame.index.name == "osm_id" or "osm_id" in frame.columns:
            frame = frame.copy()
            if frame.index.name == "osm_id":
                frame.index = pd.to_numeric(frame.index, errors="coerce").fillna(0).astype("int64")
                frame.index.name = "osm_id"
            if "osm_id" in frame.columns:
                frame["osm_id"] = pd.to_numeric(frame["osm_id"],
                                                errors="coerce").fillna(0).astype("int64")
        return real_parquet(frame, *args, **kwargs)

    def safe_records(self, *args, **kwargs):
        frame = self
        holds = [c for c in frame.columns if frame[c].dtype == object
                 and frame[c].map(lambda v: isinstance(v, (list, tuple, set, dict))).any()]
        if holds:
            frame = frame.copy()
            for col in holds:
                frame[col] = frame[col].map(
                    lambda v: ";".join(map(str, v)) if isinstance(v, (list, tuple, set))
                    else (str(v) if isinstance(v, dict) else v))
        return real_records(frame, *args, **kwargs)

    pd.DataFrame.to_parquet, pd.DataFrame.to_records = safe_parquet, safe_records
    return real_parquet, real_records


def _release_log_files(folder):
    """Let go of aequilibrae.log so the folder holding it can be removed.

    Closing a project does not detach its logger, and Windows will not unlink an open file. The
    logger is named after the project path rather than anything predictable, so the handlers are
    found by the file they write to.
    """
    target = os.path.abspath(folder)
    for logger in [logging.getLogger()] + [logging.getLogger(name) for name
                                           in list(logging.Logger.manager.loggerDict)]:
        for handler in list(getattr(logger, "handlers", [])):
            written = getattr(handler, "baseFilename", None)
            if written and os.path.abspath(written).startswith(target):
                handler.close()
                logger.removeHandler(handler)


def build_road_network(rebuild=False):
    """Open the saved car network, downloading it from OSM only if it is not there yet."""
    if rebuild and os.path.isdir(ROAD_NETWORK_DIR):
        shutil.rmtree(ROAD_NETWORK_DIR)

    if os.path.exists(os.path.join(ROAD_NETWORK_DIR, "project_database.sqlite")):
        project = Project()
        project.open(ROAD_NETWORK_DIR)
        with project.db_connection as conn:
            has_junction = "junction" in pd.read_sql(
                "PRAGMA table_info(links)", conn)["name"].tolist()
        if has_junction:
            print(f"Reopened the saved network from {ROAD_NETWORK_DIR}")
            return project
        # A tag is only captured at import time, so a cache built before `junction` was asked
        # for cannot acquire it. Fetch again rather than run without it.
        project.close()
        print("The saved network predates the junction tag; downloading again")
        _release_log_files(ROAD_NETWORK_DIR)
        try:
            shutil.rmtree(ROAD_NETWORK_DIR)
        except PermissionError as locked:
            raise PermissionError(
                f"{ROAD_NETWORK_DIR} has to be rebuilt to pick up the junction tag, but "
                f"something still holds it open. Restart the kernel, or delete the folder by "
                f"hand, and run again.") from locked

    # The district outline, smoothed: Overpass is asked for a polygon, and 7,179 sector borders
    # would make an enormous query for no extra network.
    outline = (sectors.to_crs(METRE_CRS).geometry.union_all()
               .buffer(200).simplify(2000).buffer(-200))
    outline = gpd.GeoSeries([outline], crs=METRE_CRS).to_crs("EPSG:4326").iloc[0]

    project = Project()
    project.new(ROAD_NETWORK_DIR)
    par = Parameters()

    # OSM marks roundabouts with junction=roundabout, and the importer keeps only the tags named
    # in this list - which is why `cycleway` survives and `junction` would not. The links table is
    # built from the default list when the project is created, so the column has to be added too.
    _fields = par.parameters["network"]["links"]["fields"]["one-way"]
    if not any(list(f)[0] == "junction" for f in _fields if isinstance(f, dict)):
        _fields.append({"junction": {"description": "OSM junction tag, e.g. roundabout",
                                     "osm_source": "junction", "type": "text"}})
    project.network.links.fields.add("junction", "OSM junction tag, e.g. roundabout", "TEXT")
    project.network.links.fields.save()
    par.write_back()

    osm = par.parameters["osm"]
    if osm["overpass_endpoint"].startswith("http://"):
        # The default endpoint is plain HTTP and the server now answers those with "406 Not
        # Acceptable". The same query over HTTPS is answered normally.
        osm["overpass_endpoint"] = "https://overpass-api.de/api"
        par.write_back()

    print(f"Downloading the district's car network from OSM into {ROAD_NETWORK_DIR}")
    print("  first run only, and it takes about five minutes")
    real_parquet, real_records = _patch_osm_importer()
    try:
        project.network.create_from_osm(model_area=outline, modes=["car"])
    finally:
        pd.DataFrame.to_parquet, pd.DataFrame.to_records = real_parquet, real_records
    return project


road = build_road_network()
print(f"\n{road.network.count_links():,} links, {road.network.count_nodes():,} nodes")

with road.db_connection as _conn:
    _by_type = pd.read_sql(
        "SELECT link_type, count(*) links, round(sum(distance) / 1000) km "
        "FROM links GROUP BY link_type ORDER BY links DESC", _conn)
print(f"{_by_type['km'].sum():,.0f} km of road in total")

with road.db_connection as _conn:
    _rb = pd.read_sql(
        "SELECT count(*) links, count(DISTINCT name) named, round(sum(distance)) m, "
        "sum(direction = 0) two_way FROM links WHERE junction = 'roundabout'", _conn).iloc[0]
print(f"roundabouts: {_rb['links']:,.0f} links over {_rb['m'] / 1000:,.1f} km, "
      f"{_rb['named']:,.0f} distinct names")
print(f"  {_rb['two_way']:,.0f} of them are tagged two-way, which no roundabout is\n")
print(_by_type.head(8).to_string(index=False))

## E1. Links that are not roads

The importer marks every link car-accessible, ladders and steps included. Each would otherwise
take a default speed and capacity — a ladder as a street carrying 500 vehicles an hour.

They are deleted: an import artefact, not a claim about Quito.

In [ ]:
NOT_ROADS = ["pedestrian", "bridleway", "construction", "crossing", "ladder", "rest_area",
             "bus_stop", "steps", "footway", "path", "track", "corridor", "elevator",
             "proposed", "platform", "raceway", "escalator"]


def drop_non_car_links(project):
    """Delete links no car can use. Safe to run again: on a cleaned network it finds nothing."""
    marks = ",".join("?" * len(NOT_ROADS))
    with project.db_connection as conn:
        found = pd.read_sql(
            f"SELECT link_type, count(*) links, round(sum(distance) / 1000) km FROM links "
            f"WHERE link_type IN ({marks}) GROUP BY link_type ORDER BY links DESC",
            conn, params=NOT_ROADS)
        if len(found):
            conn.execute(f"DELETE FROM links WHERE link_type IN ({marks})", NOT_ROADS)
            conn.commit()
    return found


_before_links, _before_nodes = road.network.count_links(), road.network.count_nodes()
_removed = drop_non_car_links(road)

if len(_removed):
    print(_removed.to_string(index=False))
    print(f"\nremoved {_removed['links'].sum():,} links ({_removed['km'].sum():,.0f} km)")
else:
    print("nothing to remove - the saved network is already car-only")

print(f"\ncar network: {road.network.count_links():,} links "
      f"(from {_before_links:,}), {road.network.count_nodes():,} nodes "
      f"(from {_before_nodes:,})")

with road.db_connection as _conn:
    _left = pd.read_sql("SELECT DISTINCT link_type FROM links ORDER BY link_type", _conn)
print(f"link types remaining: {', '.join(_left['link_type'])}")

## E2. The zones inside the model

The network is cached; the **zoning is not**, because it changes whenever section C changes. Each
run copies the saved network and loads the current zones into the copy.

Every zone needs a **centroid** and **connectors** — short artificial links joining it to the real
network. That is where the model's geography stops being real, and why the centroids are
population-weighted rather than geometric.

> A zone is usable only if traffic can get **both ways**. That needs a directed check: an
> undirected one passes a zone sitting downstream of one-way streets, which cars can leave but
> never enter.

> The zones go in through one `executemany` rather than `Zone.save()`, which opens a spatialite
> connection per call and aborts at around 500 zones.

> Re-running in a live kernel would leave the previous project holding the folder open. The cell
> closes it first, and says so if something else has it.

In [ ]:
# A second copy of the project, with the zones in it. TAZ_ROAD_MODEL_DIR points it
# elsewhere, so a test run can build its own without disturbing a live kernel.
ROAD_MODEL_DIR = os.environ.get("TAZ_ROAD_MODEL_DIR",
                               os.path.join(DATA_DIR, "road_model"))

# AequilibraE wants an integer zone id; ours read Z0001..Z0761.
zones["zone_number"] = zones["zone"].str.lstrip("Z").astype(int)


CONNECTORS_PER_ZONE = assume(
    "CONNECTORS_PER_ZONE", 3, "connectors joining each zone to the network",
    "one connector makes a zone's whole demand enter the network at a single node, which turns "
    "the access point into a bottleneck that is not real - the largest office zone in Inaquito "
    "was loading a one-way street at four times capacity. connect_mode picks the N nodes nearest "
    "the centroid, so this spreads the demand but does not choose where it spreads to")

RESCUE_ROADS = assume(
    "RESCUE_ROADS", ["trunk", "primary", "secondary", "tertiary"],
    "road classes a stranded zone may be joined to",
    "connect_mode only joins a centroid to a node inside its own zone, which leaves a few "
    "stranded on islands or in one-way traps; they are joined to the nearest node on a road of "
    "these classes instead, since a slightly longer connector is closer to the truth than none")


def rescue_stranded(project):
    """Join any centroid that cannot both reach and be reached to the nearest significant road.

    `connect_mode` will only attach a centroid to a node inside its own zone. That leaves two
    kinds of casualty, and they need a **directed** test to find:

    * an island - a pedestrianised block, a private estate - with no road link at all;
    * a one-way trap, where the connector lands downstream of one-way streets so cars can leave
      the zone but never enter it.

    The second is invisible to an undirected connectivity check, which is why this uses strongly
    connected components. A zone in a one-way trap is worse than useless: nothing can reach it, so
    a gravity model has to satisfy its attractions from the zone itself and it ends up sending
    100% of its trips to its own centroid.
    """
    with project.db_connection as conn:
        links = pd.read_sql(
            "SELECT link_id, a_node, b_node, direction, link_type FROM links", conn)
    with project.db_connection_spatial as conn:
        nodes = pd.read_sql("SELECT node_id, is_centroid, AsBinary(geometry) wkb FROM nodes", conn)

    points = [shapely.wkb.loads(bytes(b)) for b in nodes["wkb"]]
    nodes["lon"] = [g.x for g in points]
    nodes["lat"] = [g.y for g in points]

    # Respect one-way streets: direction 0 is both ways, 1 is a->b, -1 is b->a.
    order = {n: i for i, n in enumerate(nodes["node_id"])}
    a = links["a_node"].map(order).to_numpy()
    b = links["b_node"].map(order).to_numpy()
    way = links["direction"].to_numpy()
    src = np.concatenate([a[way >= 0], b[way <= 0]])
    dst = np.concatenate([b[way >= 0], a[way <= 0]])
    _, part = connected_components(
        coo_matrix((np.ones(len(src)), (src, dst)), shape=(len(nodes), len(nodes))),
        directed=True, connection="strong")
    nodes["component"] = part
    main = pd.Series(part).value_counts().index[0]

    centroids = nodes[nodes["is_centroid"] == 1]
    stranded = centroids[centroids["component"] != main]
    if not len(stranded):
        print("every centroid already reaches the network")
        return 0

    on_real_roads = set(links.loc[links["link_type"].isin(RESCUE_ROADS), "a_node"]) | \
        set(links.loc[links["link_type"].isin(RESCUE_ROADS), "b_node"])
    landing = nodes[(nodes["component"] == main) & (nodes["node_id"].isin(on_real_roads))
                    & (nodes["is_centroid"] != 1)]
    landing_m = gpd.GeoSeries(gpd.points_from_xy(landing["lon"], landing["lat"]),
                              crs="EPSG:4326").to_crs(METRE_CRS)
    landing_xy = np.column_stack([landing_m.x, landing_m.y])

    made, next_id = [], int(links["link_id"].max()) + 1
    print(f"{len(stranded)} centroids cannot both reach and be reached; joining them to "
          f"{'/'.join(RESCUE_ROADS)}:")
    for row in stranded.itertuples():
        here = gpd.GeoSeries(gpd.points_from_xy([row.lon], [row.lat]),
                             crs="EPSG:4326").to_crs(METRE_CRS)
        away = np.hypot(landing_xy[:, 0] - here.x.iloc[0], landing_xy[:, 1] - here.y.iloc[0])
        pick = landing.iloc[int(np.argmin(away))]
        line = LineString([(row.lon, row.lat), (pick["lon"], pick["lat"])])
        made.append((next_id, int(row.node_id), int(pick["node_id"]),
                     float(away.min()), line.wkb))
        print(f"  zone {int(row.node_id):>4} joined {away.min():>6,.0f} m away")
        next_id += 1

    with project.db_connection_spatial as conn:
        conn.executemany(
            "INSERT INTO links (link_id, a_node, b_node, direction, distance, modes, link_type, "
            "geometry) VALUES (?, ?, ?, 0, ?, 'c', 'centroid_connector', GeomFromWKB(?, 4326))",
            made)
        conn.commit()
    return len(made)


def load_zones(network_dir, zones_frame):
    """Copy the saved network and put the current zones into the copy."""
    if os.path.isdir(ROAD_MODEL_DIR):
        try:
            shutil.rmtree(ROAD_MODEL_DIR)
        except PermissionError as locked:
            raise PermissionError(
                f"{ROAD_MODEL_DIR} is held open by another process, so it cannot be "
                f"rebuilt. If a kernel has already run this section, restart it; otherwise "
                f"close any other session using the project.") from locked
    shutil.copytree(network_dir, ROAD_MODEL_DIR)

    project = Project()
    project.open(ROAD_MODEL_DIR)

    # A centroid is a node, so its id must not collide with one the network already uses.
    with project.db_connection as conn:
        lowest, highest = conn.execute("SELECT min(node_id), max(node_id) FROM nodes").fetchone()
    offset = 0 if lowest > len(zones_frame) else int(highest) + 1

    zone_rows = [(int(r.zone_number), r.geometry.wkb) for r in zones_frame.itertuples()]
    node_rows = [(offset + int(r.zone_number), r.centroid.wkb) for r in zones_frame.itertuples()]

    with project.db_connection_spatial as conn:
        conn.executemany(
            "INSERT INTO zones (zone_id, geometry) VALUES (?, ST_Multi(GeomFromWKB(?, 4326)))",
            zone_rows)
        conn.executemany(
            "INSERT INTO nodes (node_id, is_centroid, geometry) "
            "VALUES (?, 1, GeomFromWKB(?, 4326))", node_rows)
        conn.commit()
    print(f"{len(zone_rows):,} zones and centroids written "
          f"(centroid node ids start at {offset + 1:,})")

    print("Building centroid connectors to the car network...")
    project.zoning.connect_mode(mode_id="c", connectors=CONNECTORS_PER_ZONE,
                               bulk=True)
    rescue_stranded(project)
    return project


road.close()          # release the cached project before copying it
if "model" in globals():
    try:              # a previous run of this cell still holds the working copy
        model.close()
    except Exception:
        pass
model = load_zones(ROAD_NETWORK_DIR, zones)

with model.db_connection as _conn:
    _connectors = pd.read_sql(
        "SELECT count(*) n, round(avg(distance)) m FROM links "
        "WHERE link_type = 'centroid_connector'", _conn).iloc[0]
    _connected = pd.read_sql(
        "SELECT count(DISTINCT node_id) n FROM nodes WHERE is_centroid = 1 AND node_id IN "
        "(SELECT a_node FROM links WHERE link_type = 'centroid_connector' "
        " UNION SELECT b_node FROM links WHERE link_type = 'centroid_connector')",
        _conn).iloc[0, 0]

print(f"\ncentroids: {model.network.count_centroids():,} of {len(zones):,} zones")
print(f"connectors: {_connectors['n']:,.0f} links, {_connectors['m']:,.0f} m long on average")
print(f"network now: {model.network.count_links():,} links, "
      f"{model.network.count_nodes():,} nodes")

### Zones that cannot reach the network

`connect_mode` joins a centroid only to nodes **inside its own zone**. A zone with no road node
in it gets no connector, and is then invisible to the model: it produces nothing, attracts
nothing, and its residents vanish from the assignment.

In [ ]:
_isolated = len(zones) - _connected
print(f"zones with at least one connector: {_connected:,} of {len(zones):,}")
if _isolated:
    with model.db_connection as _conn:
        _has = pd.read_sql(
            "SELECT DISTINCT a_node id FROM links WHERE link_type = 'centroid_connector' "
            "UNION SELECT DISTINCT b_node FROM links WHERE link_type = 'centroid_connector'",
            _conn)["id"]
    _lost = zones[~zones["zone_number"].isin(set(_has))]
    print(f"  {len(_lost):,} zones are isolated, holding {_lost['pop'].sum():,.0f} people "
          f"({100 * _lost['pop'].sum() / zones['pop'].sum():.1f}% of the district)")
    print(_lost.nlargest(8, "pop")[["zone", "nom_par", "pop", "area_km2"]]
          .round(2).to_string(index=False))
else:
    print("  every zone reaches the network")

## E3. Speed, capacity and free-flow time

The first assumed numbers. Everything before this is measured — population counted, floor area
surveyed, zones regrouped from the census, network from OpenStreetMap. Speeds, capacities, trip
rates and a deterrence parameter are **not in the data**.

So they go through `assume()`, which records the value, the unit and why it is a guess;
`assumptions()` prints the table. Replacing one is a one-line change and a re-run.

What section E produces is **a working chain with visible parameters, not a calibrated model of
Quito**.

### What a link can carry, and how fast

OSM carries two measurements worth keeping: `maxspeed` on about 12,000 links, and a lane count on
most of the bigger roads. A value by road class fills the rest.

Capacity is **per lane**, and the per-lane rate comes from **free-flow speed** rather than the OSM
class, which is not consistent — `trunk` covers both a 90 km/h expressway and streets posted at
50. Changing a speed therefore changes a capacity.

**Connectors are deliberately unlimited.** A connector is not a road; a real capacity would make
the zone system congest rather than the traffic.

In [ ]:
SPEED_KMH = assume(
    "SPEED_KMH",
    {"motorway": 90, "motorway_link": 60, "trunk": 80, "trunk_link": 55,
     "primary": 60, "primary_link": 45, "secondary": 50, "secondary_link": 40,
     "tertiary": 40, "tertiary_link": 35, "unclassified": 30, "residential": 30,
     "living_street": 20, "service": 20, "road": 30},
    "free-flow km/h by OSM road class",
    "no speed survey; typical urban values, used only where OSM has no maxspeed tag")

LANE_CAPACITY_BY_SPEED = assume(
    "LANE_CAPACITY_BY_SPEED",
    ((80, 2000), (60, 1300), (45, 900), (35, 700), (25, 500), (0, 300)),
    "(free-flow km/h at least, vehicles per hour per lane per direction)",
    "what limits a road is whether traffic has to stop. Nothing interrupts a grade-separated "
    "road, so a lane carries what car-following allows, about 2,000/h. A signalised street "
    "discharges at a similar rate but only while the light is green, so it carries about half "
    "that. OSM does not say which a road is, and its class tag is not reliable - `trunk` covers "
    "both Simon Bolivar at 90 km/h and 575 links at 60 or below. Speed is the usable proxy: you "
    "cannot post 90 km/h on a road with traffic lights. Rates are textbook, the speed that "
    "selects them is data. The lower bands matter as much as the upper: they cover 103,184 of "
    "the 116,697 links, and lumping them together washes out the difference between a service "
    "road and a through street")

DEFAULT_LANES = assume(
    "DEFAULT_LANES",
    {"motorway": 2, "motorway_link": 1, "trunk": 2, "trunk_link": 1,
     "primary": 2, "primary_link": 1, "secondary": 2, "secondary_link": 1,
     "tertiary": 1, "tertiary_link": 1, "unclassified": 1, "residential": 1,
     "living_street": 1, "service": 1, "road": 1},
    "lanes per direction where OSM does not say",
    "used only where the lanes tag is missing; how often that happens is printed below")

CONNECTOR_SPEED, CONNECTOR_CAPACITY = assume(
    "CONNECTOR_SPEED_CAPACITY", (40, 100_000), "km/h and veh/h on centroid connectors",
    "a connector is not a road; an unlimited one keeps zone-system artefacts out of the queues")

DEFAULT_SPEED, DEFAULT_LANE_CAPACITY = assume(
    "DEFAULT_SPEED_LANE_CAPACITY", (30, 600), "km/h and veh/h per lane for an unlisted class",
    "the speed stands in for a class the table below does not list, and the check after it fails "
    "loudly if that happens. The capacity is unreachable in practice - the speed bands start at "
    "zero, so every link finds one")


def set_link_costs(project):
    """Fill speed, capacity and free-flow time.

    OSM's own `maxspeed` and lane counts are kept wherever the import found them; a value by road
    class fills in the rest. Capacity is per lane, and the rate per lane comes from the speed each
    direction runs at rather than from its class, so a 90 km/h expressway and a 50 km/h street are
    not costed alike merely because OSM files both as `trunk`.
    """
    with project.db_connection as conn:
        links = pd.read_sql(
            "SELECT link_id, link_type, lanes_ab, lanes_ba, speed_ab, speed_ba FROM links", conn)

    road = (links["link_type"] != "centroid_connector").to_numpy()
    unlisted = sorted({t for t in links.loc[road, "link_type"].dropna().unique()
                       if t not in SPEED_KMH})
    if unlisted:
        return unlisted

    fallback = links["link_type"].map(DEFAULT_LANES).fillna(1)
    speed_by_class = links["link_type"].map(SPEED_KMH).fillna(DEFAULT_SPEED)

    tagged, unreadable = {}, {}
    for side in ("ab", "ba"):
        # AequilibraE splits a two-way `lanes` tag between the directions, so halves turn up;
        # round half up and never go below one lane in a direction that exists.
        raw = pd.to_numeric(links[f"lanes_{side}"], errors="coerce")
        known = (raw > 0).to_numpy()
        lanes = np.where(known, np.maximum(1.0, np.floor(raw.fillna(0) + 0.5)), fallback)
        tagged[side] = known & road

        # Capacity follows the speed this direction actually runs at, so the order matters:
        # speed is settled first, then read back to choose the per-lane rate.
        # OSM maxspeed is free text and occasionally is not a number at all - "S/N" turns up on a
        # couple of links. Anything unreadable is treated as missing, so the class value applies.
        osm_speed = pd.to_numeric(links[f"speed_{side}"], errors="coerce")
        unreadable[side] = int((links[f"speed_{side}"].notna() & osm_speed.isna()).sum())
        links[f"speed_{side}"] = np.where(
            road, osm_speed.fillna(speed_by_class), CONNECTOR_SPEED)
        kmh = links[f"speed_{side}"].to_numpy(dtype=float)
        per_lane = np.full(len(links), float(DEFAULT_LANE_CAPACITY))
        for floor, rate in sorted(LANE_CAPACITY_BY_SPEED):
            per_lane = np.where(kmh >= floor, float(rate), per_lane)

        links[f"capacity_{side}"] = np.where(road, per_lane * lanes, CONNECTOR_CAPACITY)
        links[f"lanes_{side}"] = np.where(road, lanes, 1.0)

    rows = list(zip(links["speed_ab"].astype(float), links["speed_ba"].astype(float),
                    links["capacity_ab"].astype(float), links["capacity_ba"].astype(float),
                    links["link_id"].astype(int)))
    with project.db_connection as conn:
        conn.executemany("UPDATE links SET speed_ab = ?, speed_ba = ?, capacity_ab = ?, "
                         "capacity_ba = ? WHERE link_id = ?", rows)
        # Free-flow travel time, in MINUTES - the unit every cost in this section uses.
        conn.execute("UPDATE links SET travel_time_ab = (distance / 1000.0) / speed_ab * 60.0, "
                     "travel_time_ba = (distance / 1000.0) / speed_ba * 60.0")
        conn.commit()

    n_road = int(road.sum())
    print(f"lane counts taken from OSM: {tagged['ab'].sum():,} links one way, "
          f"{tagged['ba'].sum():,} the other, of {n_road:,} roads")
    known_either = tagged["ab"] | tagged["ba"]
    print(f"  {100 * known_either.sum() / n_road:.0f}% of roads have a lane count in at least "
          f"one direction; the rest use DEFAULT_LANES")
    if sum(unreadable.values()):
        print(f"  {sum(unreadable.values())} OSM maxspeed values were not numbers and were "
              f"treated as missing")

    print("\n  what each speed band was given:")
    _kmh = links["speed_ab"].to_numpy(dtype=float)
    _upper = np.inf
    for floor, rate in sorted(LANE_CAPACITY_BY_SPEED, reverse=True):
        inside = road & (_kmh >= floor) & (_kmh < _upper)
        top = "and up" if _upper == np.inf else f"to {int(_upper)}"
        print(f"    {floor:>3} km/h {top:<7} {rate:>5,} veh/h per lane  "
              f"{inside.sum():>7,} links")
        _upper = floor

    main = ["motorway", "trunk", "primary", "secondary", "tertiary"]
    shown = links[road & links["link_type"].isin(main)]
    if len(shown):
        print("\n  the classes that carry the load:")
        summary = shown.groupby("link_type").agg(
            links=("link_id", "size"), lanes=("lanes_ab", "mean"),
            veh_h=("capacity_ab", "mean"))
        for lt in [t for t in main if t in summary.index]:
            r = summary.loc[lt]
            print(f"    {lt:<10} {r.links:>6,.0f} links  {r.lanes:>4.1f} lanes on average  "
                  f"{r.veh_h:>6,.0f} veh/h per direction")
    return []


_from_osm = None
with model.db_connection as _conn:
    _from_osm = pd.read_sql("SELECT count(*) n FROM links WHERE speed_ab IS NOT NULL",
                            _conn).iloc[0, 0]

_unlisted = set_link_costs(model)
if _unlisted:
    raise ValueError(f"link types with no speed or capacity of their own: {_unlisted}")

with model.db_connection as _conn:
    _check = pd.read_sql(
        "SELECT count(*) links, sum(speed_ab IS NULL) no_speed, sum(capacity_ab IS NULL) no_cap, "
        "sum(travel_time_ab IS NULL) no_time, round(min(travel_time_ab), 4) fastest, "
        "round(max(travel_time_ab), 1) slowest FROM links", _conn)
print(f"speeds taken from OSM maxspeed: {_from_osm:,} links; "
      f"the other {model.network.count_links() - _from_osm:,} by road class")
print(_check.to_string(index=False))

print()
assumptions()

### Roundabouts

OSM marks a roundabout's circulating carriageway with `junction=roundabout`, and the import keeps
the tag. Costed like an ordinary link it becomes a false bottleneck: OSM records one or two
circulating lanes where there are three or four, and the model then runs it at three to four
times capacity.

A roundabout's real capacity is not a property of the carriageway at all — it is set at the
entries, by how large a gap in the circulating traffic a driver will accept. This model has no
junctions of any kind: a signalised crossroads is two links meeting at a node and costs nothing.
So the consistent treatment is for a roundabout not to be a constraint either.

Each roundabout — a connected set of tagged links, named or not — is given the capacity of the
**largest road entering it**. The ring is then never the reason an approach cannot discharge,
which is the artefact worth removing, and nothing beyond that is claimed.

This is not how roundabout capacity is really estimated. The published methods — HCM, Kimber,
gap acceptance — all compute *entry* capacity as a function of the circulating flow, and need
entry width, flare and inscribed diameter, none of which is in OSM or the cadastre.

In [ ]:
def set_roundabout_capacity(project):
    """Give each roundabout the capacity of the largest road that enters it."""
    with project.db_connection as conn:
        links = pd.read_sql(
            "SELECT link_id, a_node, b_node, name, link_type, junction, capacity_ab, "
            "capacity_ba FROM links", conn)

    circle = links[links["junction"] == "roundabout"]
    if not len(circle):
        print("no links tagged junction=roundabout")
        return pd.DataFrame()

    # A roundabout is several links in a ring. Group them by connectivity rather than by name, so
    # the unnamed ones are handled too.
    nodes = pd.unique(circle[["a_node", "b_node"]].to_numpy().ravel())
    seat = {n: i for i, n in enumerate(nodes)}
    rows = circle["a_node"].map(seat).to_numpy()
    cols = circle["b_node"].map(seat).to_numpy()
    _, part = connected_components(
        coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(nodes), len(nodes))),
        directed=False)
    ring_of_node = dict(zip(nodes, part))
    circle = circle.assign(ring=circle["a_node"].map(ring_of_node))

    roads = links[(links["junction"] != "roundabout") | links["junction"].isna()]
    roads = roads[roads["link_type"] != "centroid_connector"]

    updates, report = [], []
    for ring, part_links in circle.groupby("ring"):
        on_ring = set(part_links["a_node"]) | set(part_links["b_node"])
        # An approach is a road touching the ring but not part of it.
        approach = roads[roads["a_node"].isin(on_ring) | roads["b_node"].isin(on_ring)]
        if not len(approach):
            continue
        # The largest approach, not the total: the ring should never be the reason one
        # approach cannot discharge, and nothing justifies more than that.
        feeds = float(approach[["capacity_ab", "capacity_ba"]].max(axis=1).max())
        was = float(part_links[["capacity_ab", "capacity_ba"]].max(axis=1).mean())
        for link_id in part_links["link_id"]:
            updates.append((feeds, feeds, int(link_id)))
        report.append({"ring": int(ring), "links": len(part_links),
                       "approaches": len(approach), "was": was, "now": feeds,
                       "name": (part_links["name"].dropna().iloc[0]
                                if part_links["name"].notna().any() else "unnamed")})

    with project.db_connection as conn:
        conn.executemany(
            "UPDATE links SET capacity_ab = ?, capacity_ba = ? WHERE link_id = ?", updates)
        conn.commit()

    frame = pd.DataFrame(report)
    print(f"{len(frame)} roundabouts, {len(circle):,} links, recosted from their approaches")
    print(f"  largest approach: median {frame['now'].median():,.0f}, "
          f"biggest {frame['now'].max():,.0f} veh/h")
    print(f"  capacity per link: median {frame['was'].median():,.0f} -> "
          f"{frame['now'].median():,.0f} veh/h")
    print(f"  approaches per roundabout: median {frame['approaches'].median():.0f}")
    return frame


set_roundabout_capacity(model)

### An extra way in for the zones that need one

`connect_mode` picks the nodes nearest the centroid, which in a dense zone are all on the same
few local streets.

A zone gets one more connector, to the nearest road that can carry traffic, when its share of the
city's travel far outweighs the share of road capacity where it joins. Both are shares, so no
trip rate is needed.

Connectors are only ever **added**. Nothing is moved or removed, so no zone can lose its way on.

In [ ]:
EXTRA_CONNECTOR_MISMATCH = assume(
    "EXTRA_CONNECTOR_MISMATCH", 4.0, "how lopsided a zone must be to get another connector",
    "the ratio of a zone's share of the city's travel to the share of road capacity where it "
    "joins the network. At 4 a zone is carrying four times the traffic its access points are "
    "built for")

EXTRA_CONNECTOR_M = assume(
    "EXTRA_CONNECTOR_M", 1200, "how far an extra connector may reach",
    "far enough to find the arterial on a zone's edge, short enough that a zone still joins the "
    "network near its own people")


def add_connectors_where_needed(project, zones_frame):
    """Give a second way in to zones whose demand dwarfs the streets they join."""
    with project.db_connection as conn:
        links = pd.read_sql(
            "SELECT link_id, a_node, b_node, link_type, direction, capacity_ab, capacity_ba "
            "FROM links", conn)
    with project.db_connection_spatial as conn:
        nodes = pd.read_sql(
            "SELECT node_id, is_centroid, X(geometry) lon, Y(geometry) lat FROM nodes", conn)

    roads = links[links["link_type"] != "centroid_connector"]
    conns = links[links["link_type"] == "centroid_connector"]

    meeting = pd.concat([
        roads[["a_node", "capacity_ab"]].rename(
            columns={"a_node": "node_id", "capacity_ab": "cap"}),
        roads[["b_node", "capacity_ba"]].rename(
            columns={"b_node": "node_id", "capacity_ba": "cap"})])
    node_cap = meeting.groupby("node_id")["cap"].sum()
    nodes["cap"] = nodes["node_id"].map(node_cap).fillna(0.0)

    # Only offer nodes on a road worth reaching, that traffic can both enter and leave.
    order = {n: i for i, n in enumerate(nodes["node_id"])}
    a = roads["a_node"].map(order).to_numpy()
    b = roads["b_node"].map(order).to_numpy()
    way = roads["direction"].to_numpy()
    _, part = connected_components(
        coo_matrix((np.ones(len(a[way >= 0]) + len(b[way <= 0])),
                    (np.concatenate([a[way >= 0], b[way <= 0]]),
                     np.concatenate([b[way >= 0], a[way <= 0]]))),
                   shape=(len(nodes), len(nodes))),
        directed=True, connection="strong")
    nodes["component"] = part
    main = pd.Series(part).value_counts().index[0]

    on_big = roads[roads["link_type"].isin(RESCUE_ROADS)]
    big_nodes = set(on_big["a_node"]) | set(on_big["b_node"])
    usable = nodes[(nodes["is_centroid"] != 1) & (nodes["component"] == main)
                   & (nodes["node_id"].isin(big_nodes))]

    metres = gpd.GeoSeries(gpd.points_from_xy(nodes["lon"], nodes["lat"]),
                           crs="EPSG:4326").to_crs(METRE_CRS)
    nodes["x"], nodes["y"] = metres.x.to_numpy(), metres.y.to_numpy()
    xy = nodes.set_index("node_id")[["x", "y"]]
    cand_ids = usable["node_id"].to_numpy()
    cand_cap = usable["cap"].to_numpy()
    cand_xy = xy.loc[cand_ids].to_numpy()
    tree = cKDTree(cand_xy)

    # Where each zone joins the network now, and how much road capacity it reaches there.
    centroids = set(nodes.loc[nodes["is_centroid"] == 1, "node_id"])
    landing, held = {}, {}
    for r in conns.itertuples():
        c = r.a_node if r.a_node in centroids else r.b_node
        far = r.b_node if r.a_node in centroids else r.a_node
        landing.setdefault(c, set()).add(far)
    for c, far_nodes in landing.items():
        held[c] = float(nodes.loc[nodes["node_id"].isin(far_nodes), "cap"].sum())
    access = pd.Series(held)

    share = zones_frame.set_index("zone_number")
    trips = 0.5 * (share["pop"] / share["pop"].sum()
                   + share["activity_m2"] / share["activity_m2"].sum())
    lopsided = (trips.reindex(access.index) / (access / access.sum())).dropna()

    need = lopsided[lopsided >= EXTRA_CONNECTOR_MISMATCH].sort_values(ascending=False)
    print(f"zones whose travel outweighs their access by {EXTRA_CONNECTOR_MISMATCH}x or more: "
          f"{len(need)} of {len(access)}")
    if not len(need):
        return pd.DataFrame()

    made, next_id = [], int(links["link_id"].max()) + 1
    for centroid in need.index:
        here = xy.loc[centroid].to_numpy()
        near = tree.query_ball_point(here, EXTRA_CONNECTOR_M)
        if not near:
            continue
        near = np.array([i for i in near if cand_ids[i] not in landing[centroid]])
        if not len(near):
            continue
        best = near[np.argmax(cand_cap[near])]
        node = int(cand_ids[best])
        far = float(np.hypot(*(cand_xy[best] - here)))
        c = nodes.loc[nodes["node_id"] == centroid].iloc[0]
        n = nodes.loc[nodes["node_id"] == node].iloc[0]
        made.append((next_id, int(centroid), node, far,
                     CONNECTOR_SPEED, CONNECTOR_SPEED,
                     CONNECTOR_CAPACITY, CONNECTOR_CAPACITY,
                     (far / 1000.0) / CONNECTOR_SPEED * 60.0,
                     (far / 1000.0) / CONNECTOR_SPEED * 60.0,
                     LineString([(c["lon"], c["lat"]), (n["lon"], n["lat"])]).wkb,
                     float(access[centroid]), float(cand_cap[best]),
                     float(lopsided[centroid])))
        next_id += 1

    if not made:
        print("  none of them had a bigger road within reach; nothing added")
        return pd.DataFrame()

    with project.db_connection_spatial as conn:
        conn.executemany(
            "INSERT INTO links (link_id, a_node, b_node, direction, distance, modes, link_type, "
            "speed_ab, speed_ba, capacity_ab, capacity_ba, travel_time_ab, travel_time_ba, "
            "geometry) VALUES (?, ?, ?, 0, ?, 'c', 'centroid_connector', ?, ?, ?, ?, ?, ?, "
            "GeomFromWKB(?, 4326))", [row[:11] for row in made])
        conn.commit()

    report = pd.DataFrame([(m[1], m[2], m[3], m[11], m[12], m[13]) for m in made],
                          columns=["zone", "node", "metres", "cap_before", "cap_added",
                                   "mismatch"])
    print(f"  {len(report)} extra connectors added, median {report['metres'].median():,.0f} m")
    print("\n  the zones that got one:")
    for r in report.nlargest(10, "mismatch").itertuples():
        print(f"    zone {r.zone:>4}: {r.mismatch:>5.1f}x, reaches {r.metres:>4,.0f} m to a road "
              f"with {r.cap_added:>6,.0f} veh/h (had {r.cap_before:>6,.0f})")
    return report


add_connectors_where_needed(model, zones)

## E4. How far every zone is from every other

Free-flow travel time between every pair of zones — 761 shortest-path trees. Free-flow, not
congested: the distribution has to decide who travels where before there is traffic to delay
them.

Two things need care:

- **the diagonal is empty** — a skim says nothing about a trip that starts and ends in one zone.
  Left at zero, every zone would be infinitely attractive to itself.
- **some pairs may be unreachable** — an island of streets gives infinite cost. Counted below
  rather than quietly turned into zeros.

In [ ]:
from aequilibrae.paths import NetworkSkimming                          # noqa: E402

INTRAZONAL_SHARE = assume(
    "INTRAZONAL_SHARE", 0.5,
    "intrazonal cost as a share of the trip to the nearest other zone",
    "a skim has no diagonal; without one every zone is its own best destination and the trip "
    "matrix collapses onto itself")


def skim_free_flow(project):
    """Zone-to-zone free-flow travel time, in minutes."""
    project.network.build_graphs(modes=["c"])
    graph = project.network.graphs["c"]
    graph.prepare_graph(np.array(sorted(zones["zone_number"])))
    graph.set_graph("travel_time")
    graph.set_skimming(["travel_time"])
    graph.set_blocked_centroid_flows(True)   # no routing *through* a zone's connector

    skimming = NetworkSkimming(graph)
    skimming.execute()
    return graph, skimming.results.skims


graph, _skims = skim_free_flow(model)
cost = np.array(_skims.matrix["travel_time"], dtype=float).copy()

# Everything downstream indexes this matrix by position, so its order has to be the zone order.
# A silent mismatch here would pair each zone's trips with another zone's travel times.
SKIM_ZONES = np.array(_skims.index).astype(int)
if not np.array_equal(SKIM_ZONES, np.array(sorted(zones["zone_number"]))):
    raise ValueError("the skim is not in zone order; every cost lookup below would be wrong")
print(f"skim: {cost.shape[0]} x {cost.shape[1]} zones, index checked against the zone table")

### What the network cannot reach

The car network is not one piece: 164 components, the largest holding **99.3% of its nodes**.
That is ordinary for OSM over an area this size.

It matters only where a **centroid** lands on one of the islands. Such a zone reaches nothing and
nothing reaches it, so its residents drop out of the model. They are named rather than absorbed
into a statistic.

In [ ]:
_reachable = np.isfinite(cost) & (cost > 0)
np.fill_diagonal(_reachable, False)
_reach = _reachable.sum(axis=1)
_stranded = _reach == 0

print(f"pairs the network cannot serve: {(~_reachable).sum() - len(cost):,} of {cost.size:,} "
      f"({100 * ((~_reachable).sum() - len(cost)) / cost.size:.2f}%)")
print(f"zones reaching nowhere: {_stranded.sum():,} of {len(cost):,}")

if _stranded.any():
    _lost = zones.iloc[np.where(_stranded)[0]]
    print(f"  they hold {_lost['pop'].sum():,.0f} people "
          f"({100 * _lost['pop'].sum() / zones['pop'].sum():.2f}% of the district)")
    print(_lost[["zone", "nom_par", "pop", "area_km2"]].round(2).to_string(index=False))
    print("  their centroids sit on an island of road with no link to the network.")

_connected = ~_stranded
print(f"\nof the {_connected.sum():,} zones that are connected, each reaches "
      f"{_reach[_connected].min():,} to {_reach[_connected].max():,} of the other "
      f"{len(cost) - 1:,}")

### Filling the diagonal

A zone's internal trip is given a share of the trip to the nearest other zone. Stranded zones
keep a cost of zero; nothing is distributed to or from them anyway.

In [ ]:
_off = np.where(_reachable, cost, np.nan)
_nearest = np.full(len(cost), np.nan)
_nearest[_connected] = np.nanmin(_off[_connected], axis=1)
np.fill_diagonal(cost, np.nan_to_num(_nearest * INTRAZONAL_SHARE))

_between = cost[_reachable]
print("free-flow travel time between zones, minutes:")
for label, q in [("10th", 10), ("median", 50), ("90th", 90), ("longest", 100)]:
    print(f"  {label:>8}  {np.percentile(_between, q):6.1f}")
print(f"\nnearest other zone: median {np.nanmedian(_nearest):.1f} min, "
      f"furthest {np.nanmax(_nearest):.1f} min")
print(f"intrazonal cost: median {np.median(np.diag(cost)[_connected]):.1f} min")

## E5. How many trips each zone makes and receives

- **Productions** = population × a rate. The population is counted, so the rate is the only
  guess, and it scales the matrix without changing its shape.
- **Attractions** = floor area × a rate per class, rescaled to match the production total. Only
  the **ratios between the classes** survive that rescaling.

Residential floor area attracts too, at a low rate: a single all-purpose matrix has to carry the
journey home as well as the journey out.

Mode split is applied here rather than after distribution. Both are constants, so the matrix is
identical either way, and this keeps one matrix in play instead of two.

In [ ]:
TRIPS_PER_PERSON = assume(
    "TRIPS_PER_PERSON", 0.32, "person trips per resident per peak hour",
    "no household travel survey; a typical peak share of a 2.5-3.5 trip day")

ATTRACTION_RATES = assume(
    "ATTRACTION_RATES",
    {"mixed/commercial": 1.00, "facilities": 0.70, "heritage core": 0.60,
     "industrial": 0.25, "residential": 0.05},
    "relative pull per m2 of floor area",
    "no establishment register and no employment data; shops pull hardest, industry least. Only "
    "the ratios matter - the totals are rescaled to match productions")

CAR_SHARE = assume(
    "CAR_SHARE", 0.19, "share of person trips made by car",
    "MEASURED, not assumed: the Municipio puts private vehicles at 19% of trips today, and aims "
    "for 3% by 2042 (Secretaria de Movilidad, reported in Primicias, 8 Jan 2024). This replaces a "
    "guess of 0.35, which gave 208% more travel time than free flow and was already cut to 0.20 "
    "on the grounds that no working city shows that - a correction the official figure then "
    "confirmed to within a point")

OCCUPANCY = assume(
    "OCCUPANCY", 1.3, "people per vehicle",
    "MEASURED, not assumed: the Municipio de Quito puts the average at 1.3 people per car "
    "(Alcalde Pabel Munoz, reported in Primicias, 8 Jan 2024). Was 1.5 by guesswork")

zones["productions"] = zones["pop"] * TRIPS_PER_PERSON

_rates = pd.Series(ATTRACTION_RATES).reindex(CLASSES).fillna(0.0)
zones["pull"] = sum(zones[c] * _rates[c] for c in CLASSES)

# A gravity model needs both margins to agree. Productions come from a count and attractions from
# assumed rates, so the attractions are the side that moves.
zones["attractions"] = zones["pull"] * (zones["productions"].sum() / zones["pull"].sum())

# Person trips to vehicle trips.
VEHICLE_FACTOR = CAR_SHARE / OCCUPANCY
zones["veh_productions"] = zones["productions"] * VEHICLE_FACTOR
zones["veh_attractions"] = zones["attractions"] * VEHICLE_FACTOR

print(f"peak-hour person trips: {zones['productions'].sum():,.0f}")
print(f"  {zones['pop'].sum():,.0f} residents at {TRIPS_PER_PERSON} each")
print(f"vehicle trips: {zones['veh_productions'].sum():,.0f}")
print(f"  {CAR_SHARE:.0%} by car at {OCCUPANCY} people per vehicle "
      f"= {VEHICLE_FACTOR:.3f} vehicles per person trip")

print("\nwhere the trips start:")
print(zones.nlargest(5, "veh_productions")[["zone", "nom_par", "pop", "veh_productions"]]
      .round(0).to_string(index=False))
print("\nwhere they end:")
print(zones.nlargest(5, "veh_attractions")[["zone", "nom_par", "veh_attractions", "dominant"]]
      .round(0).to_string(index=False))

### Does this agree with the floor-area map?

Section D said the zones with the biggest circles should attract the most trips. Perfect
agreement is not the goal — the rates weight the classes differently. What would be wrong is a
zone near the top with little floor area behind it.

In [ ]:
_by_area = set(zones.nlargest(15, "activity_m2")["zone"])
_by_pull = set(zones.nlargest(15, "veh_attractions")["zone"])
print(f"in the top 15 on both counts: {len(_by_area & _by_pull)} of 15")

_moved_up = zones[zones["zone"].isin(_by_pull - _by_area)]
if len(_moved_up):
    print("\npulled into the top 15 by the rates rather than by floor area:")
    print(_moved_up[["zone", "nom_par", "dominant", "activity_m2", "veh_attractions"]]
          .round(0).to_string(index=False))
_moved_down = zones[zones["zone"].isin(_by_area - _by_pull)]
if len(_moved_down):
    print("\ndropped out of the top 15 despite the floor area:")
    print(_moved_down[["zone", "nom_par", "dominant", "activity_m2", "veh_attractions"]]
          .round(0).to_string(index=False))

## E6. The O-D matrix

A zone pair gets trips in proportion to what the origin produces, what the destination attracts,
and a **deterrence function** of the travel time between them.

`EXPO` gives `exp(-beta × minutes)`: a large `beta` keeps everyone close to home, a small one
spreads them across the city. There is no observed trip length distribution for Quito, so rather
than pick `beta` blind the notebook picks an **average trip length** — a quantity you can argue
about — and solves for the `beta` that produces it. `MEAN_TRIP_MIN` is the number to replace when
a survey turns up.

Any pair the network cannot serve gets a very large cost rather than a missing one, so it
receives almost exactly zero trips instead of dividing by nothing. The connector work leaves
none, but the guard costs nothing.

In [ ]:
from aequilibrae.distribution import GravityApplication, SyntheticGravityModel   # noqa: E402
from aequilibrae.matrix import AequilibraeMatrix                       # noqa: E402

MEAN_TRIP_MIN = assume(
    "MEAN_TRIP_MIN", 16.0, "average vehicle trip length, minutes",
    "no observed trip length distribution; this sets the deterrence parameter, which is solved "
    "for rather than guessed")

DETERRENCE = assume(
    "DETERRENCE", "EXPO", "gravity deterrence function",
    "EXPO needs one parameter where GAMMA needs two, and nothing here would justify the second")

_far = np.nanmax(cost[np.isfinite(cost)]) * 10
_impedance_values = np.where(np.isfinite(cost), cost, _far)

_vectors = (zones.set_index("zone_number")[["veh_productions", "veh_attractions"]]
            .sort_index().rename(columns={"veh_productions": "productions",
                                          "veh_attractions": "attractions"}))
_vectors["attractions"] *= _vectors["productions"].sum() / _vectors["attractions"].sum()
if not np.array_equal(_vectors.index.to_numpy().astype(int), SKIM_ZONES):
    raise ValueError("the demand vectors are not in the same order as the skim")


def gravity_matrix(beta):
    """One run of the gravity model at a given deterrence parameter."""
    impedance = AequilibraeMatrix()
    impedance.create_empty(zones=len(_vectors), matrix_names=["time"], memory_only=True)
    impedance.index[:] = _vectors.index.to_numpy()
    impedance.matrices[:, :, 0] = _impedance_values
    impedance.computational_view(["time"])

    shape = SyntheticGravityModel()
    shape.function = DETERRENCE
    shape.beta = float(beta)

    run = GravityApplication(impedance=impedance, vectors=_vectors, row_field="productions",
                             column_field="attractions", model=shape, nan_as_zero=True)
    run.apply()
    return np.array(run.output.matrix["gravity"], dtype=float)


def mean_trip_minutes(trips):
    """Trip-weighted average travel time, over the pairs the network can actually serve."""
    usable = np.isfinite(cost) & (trips > 0)
    return float((trips[usable] * cost[usable]).sum() / trips[usable].sum())


# What the model can actually produce. With no deterrence the trips follow the attractions alone,
# which fixes the longest average obtainable; with heavy deterrence everyone stays next door.
BETA_FLOOR, BETA_CEILING = 0.001, 1.0
_longest = mean_trip_minutes(gravity_matrix(BETA_FLOOR))
_shortest = mean_trip_minutes(gravity_matrix(BETA_CEILING))
print(f"average trip lengths this network and attraction pattern can produce: "
      f"{_shortest:.1f} to {_longest:.1f} min")

_target = MEAN_TRIP_MIN
if not _shortest <= _target <= _longest:
    _target = min(max(MEAN_TRIP_MIN, _shortest), _longest)
    print(f"  ** {MEAN_TRIP_MIN:.1f} min is outside that range, so the model cannot deliver it.")
    print(f"  ** Using {_target:.1f} min instead. Attractions concentrated in a central district")
    print(f"  ** pull the average down however weak the deterrence is made.")

# Bisection on beta: more deterrence means shorter trips, so the search is monotone.
_low, _high = BETA_FLOOR, BETA_CEILING
for _ in range(24):
    _mid = (_low + _high) / 2
    if mean_trip_minutes(gravity_matrix(_mid)) > _target:
        _low = _mid          # trips still too long, deter more
    else:
        _high = _mid
BETA = (_low + _high) / 2

demand = gravity_matrix(BETA)
print(f"\ndeterrence {DETERRENCE} with beta = {BETA:.4f}")
print(f"  asked for {_target:.1f} min, achieved {mean_trip_minutes(demand):.1f} min")
if BETA < 0.01:
    print("  ** beta this small means travel time barely affects where people go.")
print(f"\nvehicle trips in the matrix: {demand.sum():,.0f} "
      f"(productions asked for {_vectors['productions'].sum():,.0f})")

### Is the matrix sensible?

In [ ]:
_served = np.isfinite(cost)
print(f"cells with any trips: {(demand > 0.01).sum():,} of {demand.size:,} "
      f"({100 * (demand > 0.01).sum() / demand.size:.0f}%)")
print(f"intrazonal share: {100 * np.trace(demand) / demand.sum():.1f}% of all trips")
_self = np.diag(demand)
_share = np.divide(_self, demand.sum(axis=1), out=np.zeros_like(_self),
                   where=demand.sum(axis=1) > 0)
_stayers = np.where(_share > 0.25)[0]
print(f"  zones keeping more than a quarter of their trips at home: {len(_stayers)}")
for k in _stayers[np.argsort(-_share[_stayers])][:8]:
    z = zones[zones["zone_number"] == _vectors.index[k]].iloc[0]
    _out = cost[k].copy()
    _out[k] = np.inf
    print(f"    {z['zone']} {z['nom_par']:<20} {100 * _share[k]:3.0f}% at home, "
          f"nearest zone {np.nanmin(_out[np.isfinite(_out)]):5.1f} min, "
          f"{z['pop']:,.0f} people")
    print(f"      produces {_vectors['productions'].iloc[k]:,.0f}, "
          f"attracts {_vectors['attractions'].iloc[k]:,.0f} "
          f"(the matrix sends it {demand[:, k].sum():,.0f}); "
          f"non-residential floor area {z['activity_m2']:,.0f} m2")

_bands = [(0, 5), (5, 10), (10, 20), (20, 30), (30, 45), (45, 1e9)]
print("\nwhere the trips go, by travel time:")
for lo, hi in _bands:
    mask = _served & (cost >= lo) & (cost < hi)
    share = 100 * demand[mask].sum() / demand.sum()
    label = f"{lo}-{hi} min" if hi < 1e9 else f"over {lo} min"
    print(f"  {label:>12}  {share:5.1f}%  of trips   {'#' * int(share / 2)}")

_rows = demand.sum(axis=1)
_cols = demand.sum(axis=0)
print(f"\nrow sums against productions: correlation "
      f"{np.corrcoef(_rows, _vectors['productions'])[0, 1]:.4f}")
print(f"column sums against attractions: correlation "
      f"{np.corrcoef(_cols, _vectors['attractions'])[0, 1]:.4f}")

_top = np.dstack(np.unravel_index(np.argsort(demand, axis=None)[-8:][::-1], demand.shape))[0]
_ids = _vectors.index.to_numpy()
print("\nthe eight busiest zone pairs:")
for i, j in _top:
    o = zones[zones["zone_number"] == _ids[i]].iloc[0]
    d = zones[zones["zone_number"] == _ids[j]].iloc[0]
    kind = "within" if i == j else f"-> {d['nom_par']}"
    print(f"  {o['zone']} {o['nom_par']:<18} {kind:<22} "
          f"{demand[i, j]:6.0f} trips  {cost[i, j]:4.1f} min")

## E7. The baseline assignment

Drivers take the quickest route, their choices slow those routes down, and the solver iterates
until no one can do better by switching. The delay comes from **BPR**, which turns a link's flow
and capacity into a travel time.

The zones are the census's own units and the matrix is built from population and floor area, so
what the assignment loads is the district as the data describes it.

In [ ]:
from aequilibrae.paths import TrafficAssignment, TrafficClass                  # noqa: E402

BPR_ALPHA, BPR_BETA = assume(
    "BPR_ALPHA_BETA", (0.15, 4.0), "alpha and beta in the BPR volume-delay function",
    "no observed speed-flow data for Quito's streets; these are the values BPR is almost always "
    "published with, and the ones the scenarios notebook uses")

RGAP_TARGET, MAX_ITER = 0.001, 200


def assignment_graph(project):
    """A routing graph for the car network, costed on free-flow time."""
    project.network.build_graphs(modes=["c"])
    car = project.network.graphs["c"]
    car.prepare_graph(SKIM_ZONES)
    car.set_graph("travel_time")
    car.set_skimming(["travel_time"])
    car.set_blocked_centroid_flows(True)   # no routing *through* a zone's connector
    return car


def as_matrix(array, name="car"):
    """Wrap a zone-by-zone array as the demand matrix the solver reads."""
    holder = AequilibraeMatrix()
    holder.create_empty(zones=array.shape[0], matrix_names=[name], memory_only=True)
    holder.index[:] = SKIM_ZONES
    holder.matrices[:, :, 0] = array
    holder.computational_view([name])
    return holder


def solve(graph, matrix, label="baseline"):
    """One user-equilibrium assignment."""
    job = TrafficAssignment()
    job.add_class(TrafficClass(name="car", graph=graph, matrix=matrix))
    job.set_vdf("BPR")
    job.set_vdf_parameters({"alpha": BPR_ALPHA, "beta": BPR_BETA})
    job.set_capacity_field("capacity")
    job.set_time_field("travel_time")
    job.set_algorithm("bfw")
    job.max_iter = MAX_ITER
    job.rgap_target = RGAP_TARGET
    job.execute()
    gap, rounds = job.assignment.rgap, job.assignment.iter
    print(f"{label}: relative gap {gap:.2e} after {rounds} iterations")

    # An equilibrium solver slows down as it approaches the answer, so the gap alone does not say
    # whether more iterations would help. The curve does.
    curve = job.report()
    print("  how the gap came down:")
    for mark in (1, 5, 10, 25, 50, 100, 200, 400, 700, 1000):
        row = curve.loc[curve["iteration"] == mark, "rgap"]
        if len(row):
            print(f"    after {mark:>5} iterations: {float(row.iloc[0]):.2e}")
    trouble = curve[curve["warnings"].astype(bool) & (curve["warnings"] != "")]
    if len(trouble):
        print(f"  {len(trouble)} iterations reported a problem, first at "
              f"{int(trouble['iteration'].iloc[0])}: {trouble['warnings'].iloc[0]}")

    if gap > RGAP_TARGET:
        print(f"  ** did not reach the {RGAP_TARGET:.0e} target - it stopped at the iteration")
        print(f"  ** cap. Routes are still shifting, so link flows are indicative rather than")
        print(f"  ** settled. Raise MAX_ITER to tighten it.")
    return job.results()


car_graph = assignment_graph(model)
baseline = solve(car_graph, as_matrix(demand))

### What the network looks like carrying it

Each direction congests on its own, so **volume over capacity is a per-direction figure**. Adding
both flows and dividing by one direction's capacity would roughly double it on every two-way
street.

A street is filed under its **busier** direction; the hover panel shows both.

In [ ]:
CAR_AB, CAR_BA = "car_ab", "car_ba"     # result columns are named after the traffic class
print("result columns:", ", ".join(baseline.columns))

with model.db_connection as _conn:
    _links = pd.read_sql(
        "SELECT link_id, name, link_type, distance, capacity_ab, capacity_ba, "
        "travel_time_ab, travel_time_ba, lanes_ab, speed_ab, junction "
        "FROM links", _conn)

flows = baseline.rename_axis("link_id").reset_index().merge(_links, on="link_id", how="left")
for _side, _cap in (("ab", "capacity_ab"), ("ba", "capacity_ba")):
    flows[f"voc_{_side}"] = (flows[f"car_{_side}"].fillna(0)
                             / flows[_cap].replace(0, np.nan))
flows["voc"] = flows[["voc_ab", "voc_ba"]].max(axis=1)
flows["flow"] = flows[CAR_AB].fillna(0) + flows[CAR_BA].fillna(0)

# AequilibraE reports the same ratio itself, so the two must agree.
_gap = np.nanmax(np.abs(flows[["voc_ab", "voc_ba"]].to_numpy()
                        - flows[["VOC_AB", "VOC_BA"]].to_numpy()))
print(f"our per-direction volume/capacity vs the solver's own: "
      f"largest difference {_gap:.2e}")

streets = flows[flows["link_type"] != "centroid_connector"].copy()
loaded = streets[streets["flow"] > 0]

_veh_hours = float((flows[CAR_AB].fillna(0) * flows["Congested_Time_AB"].fillna(0)
                    + flows[CAR_BA].fillna(0)
                    * flows["Congested_Time_BA"].fillna(0)).sum() / 60)
_free_hours = float((flows[CAR_AB].fillna(0) * flows["travel_time_ab"].fillna(0)
                     + flows[CAR_BA].fillna(0)
                     * flows["travel_time_ba"].fillna(0)).sum() / 60)
print(f"\nvehicle-hours in the peak hour: {_veh_hours:,.0f}")
print(f"  the same trips on empty streets would take {_free_hours:,.0f}, "
      f"so congestion adds {100 * (_veh_hours / _free_hours - 1):.0f}%")
print(f"streets carrying traffic: {len(loaded):,} of {len(streets):,} "
      f"({100 * len(loaded) / len(streets):.0f}%)")

print("\nhow hard the loaded streets are working, by their busier direction:")
_km_total = loaded["distance"].sum() / 1000
for _lo, _hi, _label in [(0, 0.5, "under 50% - free flowing"),
                         (0.5, 0.85, "50-85% - busy"),
                         (0.85, 1.0, "85-100% - at capacity"),
                         (1.0, 1.5, "100-150% - over capacity"),
                         (1.5, 1e9, "over 150% - far beyond capacity")]:
    _band = loaded[(loaded["voc"] >= _lo) & (loaded["voc"] < _hi)]
    _km = _band["distance"].sum() / 1000
    print(f"  {_label:<34} {len(_band):>7,} links {_km:>8,.0f} km "
          f"{100 * _km / _km_total:>5.1f}%")

print("\nthe ten busiest single links, by the flow in one direction:")
_top = loaded.assign(peak=loaded[[CAR_AB, CAR_BA]].max(axis=1)).nlargest(10, "peak")
print(_top[["link_id", "name", "link_type", "peak", "capacity_ab", "voc"]]
      .rename(columns={"peak": "veh/h"}).round(2).to_string(index=False))

# Those are links, and a busy road fills the list with its own segments. Streets are the unit to
# compare against a published count, so the traffic each one carries is summed here. Vehicle-km is
# the total travel on it; the peak is the busiest point in one direction, which is the figure a
# roadside count would record.
_traffic = loaded[loaded["name"].notna() & (loaded["name"].astype(str) != "")].copy()
_traffic["veh_km"] = _traffic["flow"] * _traffic["distance"] / 1000
_traffic["one_way"] = _traffic[[CAR_AB, CAR_BA]].max(axis=1)
_streets_busy = _traffic.groupby("name").agg(
    km=("distance", "sum"), veh_km=("veh_km", "sum"), peak=("one_way", "max"),
    kind=("link_type", lambda c: c.mode().iloc[0]))
_streets_busy["km"] /= 1000

print("\nthe streets carrying the most traffic:")
print(f"  {'street':<34}{'kind':<11}{'km':>6}{'veh-km':>10}{'busiest point':>15}")
for _r in _streets_busy.nlargest(12, "veh_km").itertuples():
    print(f"  {str(_r.Index)[:33]:<34}{_r.kind:<11}{_r.km:>6.1f}{_r.veh_km:>10,.0f}"
          f"{_r.peak:>12,.0f} veh/h")

# Individual link ids are no use for looking a road up. Group the over-capacity network by street
# name, and show what the model assumed about each, so the assumptions can be checked one by one.
_over = loaded[loaded["voc"] >= 1.0]
_named = _over[_over["name"].notna() & (_over["name"].astype(str) != "")]
_anon_km = (_over["distance"].sum() - _named["distance"].sum()) / 1000
print(f"\nover-capacity network: {_over['distance'].sum() / 1000:,.0f} km, of which "
      f"{_anon_km:,.0f} km is on links OSM leaves unnamed")

_worst_streets = _named.groupby("name").agg(
    links=("link_id", "size"), km=("distance", "sum"), worst=("voc", "max"),
    typical=("voc", "median"), lanes=("lanes_ab", "mean"), kmh=("speed_ab", "mean"),
    veh_h=("capacity_ab", "mean"), kind=("link_type", lambda c: c.mode().iloc[0]))
_worst_streets["km"] /= 1000

print("\nthe streets carrying the most over-capacity length - the ones to check first:")
print(f"  {'street':<34}{'kind':<11}{'km':>6}{'lanes':>7}{'km/h':>6}"
      f"{'cap':>7}{'typical':>9}{'worst':>7}")
for _r in _worst_streets.nlargest(15, "km").itertuples():
    print(f"  {str(_r.Index)[:33]:<34}{_r.kind:<11}{_r.km:>6.1f}{_r.lanes:>7.1f}"
          f"{_r.kmh:>6.0f}{_r.veh_h:>7,.0f}{_r.typical:>9.2f}{_r.worst:>7.2f}")

# The ten roads the Agencia Metropolitana de Transito names as the city's worst, reported in
# Primicias on 8 January 2024. Occidental is the northern stretch of Mariscal Sucre, so it is
# matched under that name. Kept as data rather than prose so the comparison is redone every run.
AMT_WORST = {
    "Simon Bolivar": "Sim",
    "General Ruminahui": "Rumi",
    "Occidental (Mariscal Sucre)": "Mariscal Sucre",
    "Amazonas": "Amazonas",
    "De los Shyris": "Shyris",
    "10 de Agosto": "10 de Agosto",
    "Eloy Alfaro": "Eloy Alfaro",
    "6 de Diciembre": "6 de Diciembre",
    "Gaspar de Villarroel": "Gaspar de Villarroel",
    "Velasco Ibarra": "Velasco Ibarra",
}

print("\nagainst the ten worst roads named by the AMT (Primicias, 8 January 2024):")
print(f"  {'road':<30}{'over capacity':>14}{'typical V/C':>13}")
_named_over = _over[_over["name"].notna()]
_found = 0
for _label, _pattern in AMT_WORST.items():
    _hit = _named_over[_named_over["name"].str.contains(_pattern, case=False, regex=False)]
    if len(_hit):
        _found += 1
        print(f"  {_label:<30}{_hit['distance'].sum() / 1000:>11,.1f} km"
              f"{_hit['voc'].median():>13.2f}")
    else:
        print(f"  {_label:<30}{'not congested here':>27}")
print(f"  {_found} of {len(AMT_WORST)} appear in the model's over-capacity network")

ring = loaded[loaded["junction"] == "roundabout"]
if len(ring):
    ring_over = ring[ring["voc"] >= 1.0]
    if len(ring_over):
        print(f"\nroundabouts over capacity: {len(ring_over)} of {len(ring)} loaded links, "
              f"worst {ring_over['voc'].max():.2f}")
    else:
        print(f"\nnone of the {len(ring):,} loaded roundabout links is over capacity")

print("\nand the sharpest single points, wherever they are:")
for _r in _named.nlargest(8, "voc").itertuples():
    print(f"  {str(_r.name)[:33]:<34}{_r.link_type:<11}link {_r.link_id:<7}"
          f"{_r.lanes_ab:>4.0f} lanes {_r.speed_ab:>4.0f} km/h  "
          f"cap {_r.capacity_ab:>6,.0f}  V/C {_r.voc:>5.2f}")

## E8. The congestion map

Colour is volume over capacity of the busier direction, in the bands above. Only the classes that
carry through-traffic are drawn — trunk to tertiary with their ramps — because residential and
service links are most of the network, carry little of the load, and would fill the map with grey.

Read it as **where the model puts pressure**, not as a congestion forecast.

In [ ]:
CONGESTION_BANDS = [(0.00, 0.50, "#2c7bb6", "under 50% - free flowing"),
                    (0.50, 0.85, "#8cc4de", "50-85% - busy"),
                    (0.85, 1.00, "#fdae61", "85-100% - at capacity"),
                    (1.00, 1.50, "#d7191c", "100-150% - over capacity"),
                    (1.50, 1e9, "#7d0018", "over 150% - far beyond capacity")]

MAP_ROAD_TYPES = ["motorway", "motorway_link", "trunk", "trunk_link",
                  "primary", "primary_link", "secondary", "secondary_link",
                  "tertiary", "tertiary_link"]

# Thicker for the bigger roads, so the map reads as a hierarchy at low zoom.
LINE_WEIGHT = {"motorway": 3.6, "trunk": 3.2, "primary": 2.6,
               "secondary": 2.0, "tertiary": 1.4}
LINK_SIMPLIFY_M = 8


def link_geometry(project, keep):
    """Geometry for the links we are about to draw, simplified for the browser."""
    # AsBinary and the other spatialite functions live only on the spatial connection.
    with project.db_connection_spatial as conn:
        geo = pd.read_sql("SELECT link_id, AsBinary(geometry) wkb FROM links", conn)
    geo = geo[geo["link_id"].isin(set(keep))].copy()
    geo["geometry"] = [shapely.wkb.loads(bytes(b)) for b in geo["wkb"]]
    geo = gpd.GeoDataFrame(geo.drop(columns="wkb"), geometry="geometry", crs="EPSG:4326")
    geo["geometry"] = geo.to_crs(METRE_CRS).simplify(LINK_SIMPLIFY_M).to_crs("EPSG:4326")
    geo["geometry"] = shapely.set_precision(geo.geometry.values, COORD_GRID)
    return geo


def map_congestion(frame, filename="5_congestion.html"):
    """Loaded streets coloured by how far past capacity their busier direction runs."""
    drawn = frame[frame["link_type"].isin(MAP_ROAD_TYPES) & (frame["flow"] > 0)].copy()
    drawn = drawn.merge(link_geometry(model, drawn["link_id"]), on="link_id", how="inner")
    drawn = gpd.GeoDataFrame(drawn, geometry="geometry", crs="EPSG:4326")
    drawn["base"] = drawn["link_type"].str.replace("_link", "", regex=False)
    drawn["weight"] = drawn["base"].map(LINE_WEIGHT).fillna(1.4)
    # A ramp is drawn thinner than the road it serves.
    drawn.loc[drawn["link_type"].str.endswith("_link"), "weight"] *= 0.6
    print(f"drawing {len(drawn):,} links, {drawn['distance'].sum() / 1000:,.0f} km")

    def rows_for(r):
        rows = [("class", r.link_type), ("length", f"{r.distance:,.0f} m")]
        for side, label in (("ab", "A&rarr;B"), ("ba", "B&rarr;A")):
            veh, voc = getattr(r, f"car_{side}"), getattr(r, f"voc_{side}")
            if pd.notna(veh) and veh > 0:
                rows.append((f"{label} flow", f"{veh:,.0f} veh/h"))
                rows.append((f"{label} of capacity", f"{100 * voc:,.0f}%"))
        free, now = r.travel_time_ab, r.Congested_Time_AB
        if pd.notna(free) and pd.notna(now) and free > 0:
            rows.append(("time A&rarr;B", f"{now:.1f} min, {now / free:.1f}&times; free flow"))
        return rows

    m = folium.Map(tiles="CartoDB positron")
    m.get_root().header.add_child(folium.Element(TIP_CSS))
    minx, miny, maxx, maxy = drawn.total_bounds
    m.fit_bounds([[miny, minx], [maxy, maxx]])

    # One layer per band, quiet roads first so the jammed ones end up drawn on top.
    for lo, hi, colour, label in CONGESTION_BANDS:
        band = drawn[(drawn["voc"] >= lo) & (drawn["voc"] < hi)].copy()
        if band.empty:
            continue
        named = band["name"].fillna("unnamed street").replace("", "unnamed street")
        band["tip"] = [tip_html(nm, rows_for(r), tag=label.split(" - ")[-1], tag_color=colour)
                       for nm, r in zip(named, band.itertuples())]
        folium.GeoJson(
            band[["geometry", "tip", "weight"]],
            name=f"{label}  ({len(band):,})",
            style_function=lambda f, c=colour: {
                "color": c, "weight": f["properties"]["weight"], "opacity": 0.9},
            tooltip=panel_tooltip(),
        ).add_to(m)

    BOX = ("position:fixed;bottom:24px;left:24px;z-index:9999;background:white;"
           "padding:9px 12px;border:1px solid #999;border-radius:6px;"
           "font-family:sans-serif;font-size:13px;line-height:1.45;max-width:285px;")
    PANEL = ("position:fixed;top:20px;right:20px;z-index:9999;background:white;"
             "padding:9px 12px;border:1px solid #999;border-radius:6px;"
             "font-family:sans-serif;font-size:13px;line-height:1.5;max-width:265px;")
    swatches = "".join(
        f'<div><span style="display:inline-block;width:16px;height:4px;background:{c};'
        f'margin:0 6px 3px 0;vertical-align:middle;"></span>{lab}</div>'
        for _, _, c, lab in CONGESTION_BANDS)
    m.get_root().html.add_child(folium.Element(
        f'<div style="{BOX}"><b>Volume over capacity</b>{swatches}</div>'))

    over_km = drawn[drawn["voc"] >= 1.0]["distance"].sum() / 1000
    all_km = drawn["distance"].sum() / 1000
    m.get_root().html.add_child(folium.Element(
        f'<div style="{PANEL}"><b>Peak-hour car traffic</b>'
        f'<div>{demand.sum():,.0f} vehicle trips over {len(zones):,} zones</div>'
        f'<div>{_veh_hours:,.0f} vehicle-hours, {100 * (_veh_hours / _free_hours - 1):.0f}% more '
        f'than the same trips on empty streets</div>'
        f'<div>{over_km:,.0f} of {all_km:,.0f} km drawn are over capacity</div>'
        f'</div>'))

    path = os.path.join(MAPS_DIR, filename)
    m.save(path)
    print(f"Map saved: {filename}  ({os.path.getsize(path) / 1e6:.1f} MB)"
          f"\n  location: {os.path.abspath(path)}")
    return m


map_congestion(streets)

### Delay, and what it comes to over a year

Delay is the vehicle-hours spent on the loaded network, less what the same trips would have taken
on empty streets. **Per trip, that is a figure the model earns.**

Turning it into hours a year needs a guess about how often a person makes such a trip, registered
like any other. The comparison with a published figure is rough on both sides: ours averages
every trip, short ones included, while a published figure usually describes a commuter.

In [ ]:
PEAK_TRIPS_PER_DAY = assume(
    "PEAK_TRIPS_PER_DAY", 2, "peak-hour trips a person makes on a working day",
    "one out and one back; nothing in the data says so")

WORKING_DAYS = assume(
    "WORKING_DAYS", 250, "working days a year",
    "a common figure for a five-day year less holidays; not specific to Quito")

_delay_h = _veh_hours - _free_hours
_per_trip = _delay_h / demand.sum() * 60
_per_year = PEAK_TRIPS_PER_DAY * WORKING_DAYS * _per_trip / 60

print(f"delay in the peak hour: {_delay_h:,.0f} vehicle-hours "
      f"({_veh_hours:,.0f} congested less {_free_hours:,.0f} at free flow)")
print(f"  over {demand.sum():,.0f} trips, that is {_per_trip:.2f} minutes each")
print(f"  at {PEAK_TRIPS_PER_DAY} peak trips a day for {WORKING_DAYS} days: "
      f"{_per_year:.0f} hours a year")
print(f"  the Agencia Metropolitana de Transito puts it at 63 "
      f"(Primicias, 8 January 2024)")

# The same number whether counted per vehicle or per person: occupancy multiplies the delay and
# the trips alike, so it cancels.
print(f"\nfor comparison, at other guesses about how often people travel:")
for _n, _d in ((2, 220), (2, 250), (2, 280), (3, 250)):
    print(f"  {_n} trips a day for {_d} days: {_n * _d * _per_trip / 60:.0f} hours a year")

### How much of this is right

Two published figures, neither of which the model was fitted to.

- **Which corridors carry the load.** The Agencia Metropolitana de Tránsito names these ten as
  the city's worst (*Primicias*, 8 January 2024):

  | | | |
  |---|---|---|
  | Simón Bolívar | Amazonas | Eloy Alfaro |
  | General Rumiñahui | De los Shyris | 6 de Diciembre |
  | Occidental (Mariscal Sucre) | 10 de Agosto | Gaspar de Villarroel |
  | | | Velasco Ibarra |

  The table above says which of them this model finds congested, and by how much. From census
  population, cadastral floor area and OpenStreetMap geometry alone, it puts **Simón Bolívar
  first**, as AMT does.
- **How much time congestion costs.** AMT puts it at 63 hours a year per person; the model's
  figure is worked out above.

`CAR_SHARE` and `OCCUPANCY` come from the Municipio. Everything else is measured, or in the
assumptions table.

### What it does not capture

- **BPR is not physical above capacity.** Flow cannot really exceed capacity — traffic queues and
  spills back instead. Rumiñahui's worst link runs at 2.55 times capacity, which BPR turns into
  **7.3 times free-flow time** against a reported 4.5. Read the totals and the pattern, not the
  delay on any one link.
- **How many ways into a zone is a guess.** Three connectors each, four for 28 zones. Going from
  one connector to that moved congestion from 41% to 35%, and hours lost per person from 57 to
  49. Which is closer to the truth is unsettled: Quito's local streets really are enclosed within
  *ciudadelas* and gated developments, so few ways into a district may be a feature of the city.
- **Roundabouts carry their largest approach, and their entries are not modelled.** The ring can
  no longer pinch a single approach below what that road already carries, but it can still be
  over capacity where two approaches converge on one arc - the count is printed above. What is
  missing is the entry process: a real roundabout's capacity *falls* as its circulating flow
  rises, which is why it queues. Modelling that needs entry geometry nobody has published, and
  OSM's circulating lane counts are wrong here anyway - 688 of the 726 links are tagged two-way,
  which no roundabout is.
- **Residential streets are used as short cuts.** The sharpest single points are small streets
  like Hacienda Maria and La Condamine, carrying three times their capacity while serving no
  zone. Nothing stops them: there are no turn penalties, no junction delay and no traffic
  calming, so a back street is a free short cut.
- **Some streets congest for reasons a static assignment cannot see.** De los Shyris and Gaspar
  de Villarroel are on AMT's list and quiet here. They are in the network, drivable, with zones
  attached — but what fills them is buses pulling in, parked cars and deliveries, none of which
  this model represents.